In [ ]:
# -*- coding: utf-8 -*-
"""
Пайплайн расчёта СРПВ (PWV) — одиночный файл
МК: MSP430i2040  |  fs = 488.28 Гц (SMCLK=2.048 МГц, OSR=256, AVG=16)

Каналы в CSV:
  col 0 — счётчик (не используется для времени)
  col 1 — плетизмограмма грудь (Hall 1)
  col 2 — плетизмограмма рука  (Hall 2)
  col 3 — Hall 3, мусор (игнорируем)
  col 4 — ЭКГ
"""

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, savgol_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── НАСТРОЙКИ ───────────────────────────────────────────────────────────────
DATA_FILE    = 'data4ch_0_7.csv'   # входной файл
DISTANCE_M   = 0.5                 # расстояние грудь–запястье, м
FS           = 488.28              # частота дискретизации из прошивки МК

# Детекция R-пиков
TKEO_FACTOR  = 0.4
MIN_RR_SEC   = 0.4                 # минимальный RR-интервал (согласован)

# Поиск foot
FOOT_START_MS = 50
FOOT_END_MS   = 400

# Допустимые границы PTT
PTT_MIN_MS   = 20
PTT_MAX_MS   = 200


# ═════════════════════════════════════════════════════════════════════════════
# 1. УТИЛИТЫ ФИЛЬТРАЦИИ
# ═════════════════════════════════════════════════════════════════════════════

def butter_bandpass(data, low, high, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, min(high / nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)


def butter_lowpass(data, cutoff, fs, order=3):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low')
    return filtfilt(b, a, data)


def fix_zeros(sig):
    """
    Замена нулевых отсчётов (артефакты записи) линейной интерполяцией.
    Корректно обрабатывает нули на границах сигнала.
    """
    sig = sig.copy().astype(float)
    zero_mask = (sig == 0)

    if not zero_mask.any():
        return sig

    # Индексы «хороших» отсчётов
    good_idx = np.where(~zero_mask)[0]
    if len(good_idx) == 0:
        return sig  # весь сигнал — нули, ничего не делаем

    all_idx = np.arange(len(sig))
    # np.interp автоматически клэмпит значения за границами к крайним точкам
    sig[zero_mask] = np.interp(all_idx[zero_mask], good_idx, sig[good_idx])
    return sig


# ═════════════════════════════════════════════════════════════════════════════
# 2. ПРЕДОБРАБОТКА ЭКГ
# ═════════════════════════════════════════════════════════════════════════════

def preprocess_ecg(raw, fs):
    """Полосовая фильтрация + z-нормировка."""
    ecg = butter_bandpass(raw, 0.5, min(40.0, fs * 0.45), fs)
    return (ecg - ecg.mean()) / (ecg.std() + 1e-8)


# ═════════════════════════════════════════════════════════════════════════════
# 3. ДЕТЕКЦИЯ R-ПИКОВ (TKEO)
# ═════════════════════════════════════════════════════════════════════════════

def detect_rpeaks(ecg, fs, min_rr_sec=MIN_RR_SEC, tkeo_factor=TKEO_FACTOR):
    """
    Детекция R-пиков через оператор Тейгера–Кайзера (TKEO).
    Автоматически обрабатывает инвертированный ЭКГ.
    """
    # Определяем полярность
    pos = np.max(ecg) - np.median(ecg)
    neg = np.median(ecg) - np.min(ecg)
    inverted = neg > pos
    ecg_proc = -ecg if inverted else ecg

    # TKEO: x[n]^2 - x[n-1]*x[n+1]
    tkeo = ecg_proc[1:-1] ** 2 - ecg_proc[:-2] * ecg_proc[2:]
    tkeo = np.insert(tkeo, 0, 0.0)

    # Сглаживание Савицкого–Голая
    win = max(3, int(0.05 * fs) | 1)
    tkeo_sm = savgol_filter(tkeo, window_length=win, polyorder=2)

    # Порог и минимальное расстояние между пиками
    thresh   = tkeo_factor * np.percentile(tkeo_sm, 98)
    min_dist = int(min_rr_sec * fs)
    candidates, _ = find_peaks(tkeo_sm, height=thresh, distance=min_dist)

    # Уточнение позиции пика в исходном ЭКГ
    half_win = max(1, int(0.03 * fs))
    peak_indices, peak_heights = [], []
    for p in candidates:
        lo = max(0, p - half_win)
        hi = min(len(ecg), p + half_win + 1)
        if inverted:
            local = lo + np.argmin(ecg[lo:hi])
        else:
            local = lo + np.argmax(ecg[lo:hi])

        # Высота относительно локального базиса
        win_bl = int(1.0 * fs)
        bl_s = max(0, local - win_bl)
        bl_e = min(len(ecg), local + win_bl)
        baseline = np.median(ecg[bl_s:bl_e])
        peak_indices.append(local)
        peak_heights.append(abs(ecg[local] - baseline))

    peak_indices = np.array(peak_indices)
    peak_heights = np.array(peak_heights)

    if len(peak_heights) < 2:
        return peak_indices

    # Отбрасываем мелкие пики (< 50% медианы высоты)
    keep = peak_heights >= 0.5 * np.median(peak_heights)
    return np.unique(peak_indices[keep])


# ═════════════════════════════════════════════════════════════════════════════
# 4. ДЕТЕКЦИЯ FOOT НА ПЛЕТИЗМОГРАММЕ
# ═════════════════════════════════════════════════════════════════════════════

def detect_feet_chest(signal, rpeaks, fs,
                      start_ms=FOOT_START_MS, end_ms=FOOT_END_MS):
    """
    Для каждого R-пика ищем foot (начало волны) в сигнале груди.
    Foot = минимум в окне [start_ms, 80% до систолического пика].
    """
    start_dt = int(start_ms * fs / 1000)
    end_dt   = int(end_ms   * fs / 1000)
    feet = []
    for r in rpeaks:
        lo, hi = r + start_dt, r + end_dt
        if hi >= len(signal):
            continue
        seg = signal[lo:hi]
        if len(seg) < 3:
            continue
        peak_rel    = np.argmax(seg)
        search_end  = max(1, int(peak_rel * 0.8))
        foot_rel    = np.argmin(seg[:search_end])
        feet.append(lo + foot_rel)
    return np.array(feet, dtype=int)


def detect_feet_arm(signal, feet_chest, fs,
                    ptt_min_ms=PTT_MIN_MS, ptt_max_ms=PTT_MAX_MS):
    """
    Для каждого foot груди ищем соответствующий foot на руке
    в окне PTT_MIN … PTT_MAX после foot груди.
    Возвращает массив индексов (-1 если пара не найдена).
    """
    start_dt = int(ptt_min_ms * fs / 1000)
    end_dt   = int(ptt_max_ms * fs / 1000)
    feet_arm = []
    for fc in feet_chest:
        lo, hi = fc + start_dt, fc + end_dt
        if hi >= len(signal) or lo < 0:
            feet_arm.append(-1)
            continue
        seg = signal[lo:hi]
        if len(seg) < 3:
            feet_arm.append(-1)
            continue
        feet_arm.append(lo + np.argmin(seg))
    return np.array(feet_arm, dtype=int)


# ═════════════════════════════════════════════════════════════════════════════
# 5. РАСЧЁТ PTT / PWV
# ═════════════════════════════════════════════════════════════════════════════

def compute_pwv(feet_chest, feet_arm, fs, distance_m):
    """
    Вычисляет PTT и PWV для валидных пар.
    Применяет MAD-фильтр (z < 3.5) для удаления выбросов.
    Возвращает: ptt_arr, pwv_arr, fc_valid, fa_valid (все без выбросов).
    """
    valid = feet_arm >= 0
    fc_v  = feet_chest[valid]
    fa_v  = feet_arm[valid]

    if len(fc_v) == 0:
        return np.array([]), np.array([]), fc_v, fa_v

    ptt = (fa_v - fc_v) / fs
    # Убираем физически невозможные значения (отрицательный PTT)
    positive = ptt > 0
    fc_v, fa_v, ptt = fc_v[positive], fa_v[positive], ptt[positive]

    if len(ptt) == 0:
        return np.array([]), np.array([]), fc_v, fa_v

    pwv = distance_m / ptt

    # MAD-фильтр выбросов
    med = np.median(ptt)
    mad = np.median(np.abs(ptt - med))
    if mad > 0:
        z    = np.abs(ptt - med) / (mad * 1.4826)
        keep = z < 3.5
    else:
        keep = np.ones(len(ptt), dtype=bool)

    return ptt[keep], pwv[keep], fc_v[keep], fa_v[keep]


# ═════════════════════════════════════════════════════════════════════════════
# 6. ВИЗУАЛИЗАЦИЯ
# ═════════════════════════════════════════════════════════════════════════════

def plot_results(time, ecg, rpeaks, chest, feet_chest,
                 wrist, feet_arm_valid, fc_final, fa_final, ptt_val, pwv_val):
    """Интерактивный трёхпанельный график."""

    pwv_med  = float(np.median(pwv_val)) if len(pwv_val) else float('nan')
    ptt_med  = float(np.median(ptt_val)) if len(ptt_val) else float('nan')
    title    = (f'СРПВ = {pwv_med:.2f} м/с  |  '
                f'PTT = {ptt_med*1000:.1f} мс  |  '
                f'Пар: {len(ptt_val)}')

    fig = make_subplots(
        rows=3, cols=1, shared_xaxes=True,
        subplot_titles=('ЭКГ + R-пики', 'Грудь + foot', 'Рука + foot'),
        vertical_spacing=0.07)

    # ЭКГ
    fig.add_trace(go.Scattergl(x=time, y=ecg,
        name='ЭКГ', line=dict(color='royalblue', width=0.8)), row=1, col=1)
    fig.add_trace(go.Scattergl(x=time[rpeaks], y=ecg[rpeaks],
        mode='markers', name='R-пики',
        marker=dict(color='red', size=7, symbol='x')), row=1, col=1)

    # Грудь
    fig.add_trace(go.Scattergl(x=time, y=chest,
        name='Грудь', line=dict(color='seagreen', width=0.8)), row=2, col=1)
    fig.add_trace(go.Scattergl(x=time[feet_chest], y=chest[feet_chest],
        mode='markers', name='Foot грудь',
        marker=dict(color='darkgreen', size=8, symbol='circle-open')), row=2, col=1)

    # Рука
    fig.add_trace(go.Scattergl(x=time, y=wrist,
        name='Рука', line=dict(color='mediumpurple', width=0.8)), row=3, col=1)
    if len(fa_final) > 0:
        fa_plot = fa_final[fa_final >= 0]
        fig.add_trace(go.Scattergl(x=time[fa_plot], y=wrist[fa_plot],
            mode='markers', name='Foot рука',
            marker=dict(color='darkviolet', size=8, symbol='circle-open')), row=3, col=1)

    # PTT-стрелки (первые 5 для наглядности)
    for i in range(min(5, len(ptt_val))):
        fc, fa = fc_final[i], fa_final[i]
        if fa < 0:
            continue
        fig.add_trace(go.Scattergl(
            x=[time[fc], time[fa]],
            y=[chest[fc], wrist[fa]],
            mode='lines+markers',
            line=dict(color='black', dash='dot', width=1),
            marker=dict(size=5, color='black'),
            showlegend=(i == 0),
            name=f'PTT={ptt_val[i]*1000:.0f} мс'), row=2, col=1)

    fig.update_layout(height=820, title=title, hovermode='x unified')
    fig.update_xaxes(rangeslider=dict(visible=True), row=3, col=1)
    fig.update_xaxes(title_text='Время, с', row=3, col=1)
    fig.update_yaxes(title_text='z-score', row=1, col=1)
    fig.update_yaxes(title_text='Амплитуда', row=2, col=1)
    fig.update_yaxes(title_text='Амплитуда', row=3, col=1)
    fig.show()


# ═════════════════════════════════════════════════════════════════════════════
# 7. ОСНОВНАЯ ФУНКЦИЯ
# ═════════════════════════════════════════════════════════════════════════════

def process_file(filepath=DATA_FILE, plot=True, verbose=True):
    """
    Полный пайплайн для одного CSV-файла.
    Возвращает dict с результатами.
    """
    def log(msg):
        if verbose:
            print(msg)

    log(f'\n{"="*55}')
    log(f'Файл: {filepath}')

    # ── Загрузка ──────────────────────────────────────────
    df = pd.read_csv(filepath, header=None)
    log(f'Строк: {len(df)}, столбцов: {df.shape[1]}')

    # Правильная временная ось: fs из прошивки
    time         = np.arange(len(df)) / FS
    ch_chest_raw = df.iloc[:, 1].values.astype(float)
    ch_wrist_raw = df.iloc[:, 2].values.astype(float)
    ecg_raw      = df.iloc[:, 4].values.astype(float)
    log(f'Длина записи: {time[-1]:.1f} с')

    # ── Предобработка ─────────────────────────────────────
    ecg_filt    = preprocess_ecg(ecg_raw, FS)
    ch_chest_lp = butter_lowpass(fix_zeros(ch_chest_raw), cutoff=10.0, fs=FS)
    ch_wrist_lp = butter_lowpass(fix_zeros(ch_wrist_raw), cutoff=10.0, fs=FS)

    # ── R-пики ────────────────────────────────────────────
    rpeaks = detect_rpeaks(ecg_filt, FS, MIN_RR_SEC, TKEO_FACTOR)
    log(f'R-пиков найдено: {len(rpeaks)}')
    if len(rpeaks) < 3:
        log('[!] Слишком мало R-пиков, обработка невозможна.')
        return None

    # ── Foot ──────────────────────────────────────────────
    feet_chest = detect_feet_chest(ch_chest_lp, rpeaks, FS,
                                   FOOT_START_MS, FOOT_END_MS)
    feet_arm   = detect_feet_arm(ch_wrist_lp, feet_chest, FS,
                                 PTT_MIN_MS, PTT_MAX_MS)
    log(f'Foot груди: {len(feet_chest)},  пар с рукой: {(feet_arm >= 0).sum()}')

    # ── PTT / PWV ─────────────────────────────────────────
    ptt_val, pwv_val, fc_final, fa_final = compute_pwv(
        feet_chest, feet_arm, FS, DISTANCE_M)

    if len(ptt_val) == 0:
        log('[!] Валидных PTT не найдено.')
        return None

    pwv_med = float(np.median(pwv_val))
    ptt_med = float(np.median(ptt_val))

    log(f'\nРезультаты:')
    log(f'  Валидных пар (после MAD-фильтра): {len(ptt_val)}')
    log(f'  PTT (медиана):  {ptt_med*1000:.1f} мс')
    log(f'  PTT (std):      {np.std(ptt_val)*1000:.1f} мс')
    log(f'  PWV (медиана):  {pwv_med:.2f} м/с')
    log(f'  PWV (std):      {np.std(pwv_val):.2f} м/с')

    result = dict(
        file       = str(filepath),
        duration_s = round(float(time[-1]), 1),
        n_rpeaks   = int(len(rpeaks)),
        n_valid    = int(len(ptt_val)),
        ptt_ms     = round(ptt_med * 1000, 1),
        ptt_std_ms = round(float(np.std(ptt_val)) * 1000, 1),
        pwv_ms     = round(pwv_med, 2),
        pwv_std    = round(float(np.std(pwv_val)), 2),
    )

    if plot:
        plot_results(time, ecg_filt, rpeaks,
                     ch_chest_lp, feet_chest,
                     ch_wrist_lp, fa_final,
                     fc_final, fa_final, ptt_val, pwv_val)

    return result


# ─── ТОЧКА ВХОДА ─────────────────────────────────────────────────────────────
if __name__ == '__main__':
    res = process_file(DATA_FILE, plot=True)
    if res:
        print('\nИтог:', res)


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# -*- coding: utf-8 -*-
"""
Пайплайн расчёта СРПВ (PWV) — одиночный файл
МК: MSP430i2040  |  fs = 488.28 Гц (SMCLK=2.048 МГц, OSR=256, AVG=16)

Каналы в CSV:
  col 0 — счётчик (не используется для времени)
  col 1 — плетизмограмма грудь (Hall 1)
  col 2 — плетизмограмма рука  (Hall 2)
  col 3 — Hall 3, мусор (игнорируем)
  col 4 — ЭКГ
"""

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, savgol_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── НАСТРОЙКИ ───────────────────────────────────────────────────────────────
DATA_FILE    = 'data4ch_0_5.csv'   # входной файл
DISTANCE_M   = 0.5                 # расстояние грудь–запястье, м
FS           = 488.28              # частота дискретизации из прошивки МК

# Детекция R-пиков
TKEO_FACTOR  = 0.6
MIN_RR_SEC   = 0.4                 # минимальный RR-интервал (согласован)

# Поиск foot
FOOT_START_MS = 50
FOOT_END_MS   = 400

# Допустимые границы PTT
PTT_MIN_MS   = 20
PTT_MAX_MS   = 200


# ═════════════════════════════════════════════════════════════════════════════
# 1. УТИЛИТЫ ФИЛЬТРАЦИИ
# ═════════════════════════════════════════════════════════════════════════════

def butter_bandpass(data, low, high, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, min(high / nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)


def butter_lowpass(data, cutoff, fs, order=3):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low')
    return filtfilt(b, a, data)


def butter_bandpass_pleth(data, low=0.5, high=10.0, fs=FS, order=3):
    """
    Полосовой фильтр для плетизмограмм.
    Убирает DC-дрейф и медленные ступеньки (< 0.5 Гц)
    и высокочастотный шум (> 10 Гц).
    Оставляет только физиологический диапазон пульсовых волн.
    """
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, min(high / nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)


def fix_zeros(sig):
    """
    Замена нулевых отсчётов (артефакты записи) линейной интерполяцией.
    Корректно обрабатывает нули на границах сигнала.
    """
    sig = sig.copy().astype(float)
    zero_mask = (sig == 0)

    if not zero_mask.any():
        return sig

    # Индексы «хороших» отсчётов
    good_idx = np.where(~zero_mask)[0]
    if len(good_idx) == 0:
        return sig  # весь сигнал — нули, ничего не делаем

    all_idx = np.arange(len(sig))
    # np.interp автоматически клэмпит значения за границами к крайним точкам
    sig[zero_mask] = np.interp(all_idx[zero_mask], good_idx, sig[good_idx])
    return sig


# ═════════════════════════════════════════════════════════════════════════════
# 2. ПРЕДОБРАБОТКА ЭКГ
# ═════════════════════════════════════════════════════════════════════════════

def preprocess_ecg(raw, fs):
    """Полосовая фильтрация + z-нормировка."""
    ecg = butter_bandpass(raw, 0.5, min(40.0, fs * 0.45), fs)
    return (ecg - ecg.mean()) / (ecg.std() + 1e-8)


# ═════════════════════════════════════════════════════════════════════════════
# 3. ДЕТЕКЦИЯ R-ПИКОВ (TKEO)
# ═════════════════════════════════════════════════════════════════════════════

def detect_rpeaks(ecg, fs, min_rr_sec=MIN_RR_SEC, tkeo_factor=TKEO_FACTOR):
    """
    Детекция R-пиков через оператор Тейгера–Кайзера (TKEO).
    Автоматически обрабатывает инвертированный ЭКГ.
    """
    # Определяем полярность
    pos = np.max(ecg) - np.median(ecg)
    neg = np.median(ecg) - np.min(ecg)
    inverted = neg > pos
    ecg_proc = -ecg if inverted else ecg

    # TKEO: x[n]^2 - x[n-1]*x[n+1]
    tkeo = ecg_proc[1:-1] ** 2 - ecg_proc[:-2] * ecg_proc[2:]
    tkeo = np.insert(tkeo, 0, 0.0)

    # Сглаживание Савицкого–Голая
    win = max(3, int(0.05 * fs) | 1)
    tkeo_sm = savgol_filter(tkeo, window_length=win, polyorder=2)

    # Порог и минимальное расстояние между пиками
    thresh   = tkeo_factor * np.percentile(tkeo_sm, 98)
    min_dist = int(min_rr_sec * fs)
    candidates, _ = find_peaks(tkeo_sm, height=thresh, distance=min_dist)

    # Уточнение позиции пика в исходном ЭКГ
    half_win = max(1, int(0.03 * fs))
    peak_indices, peak_heights = [], []
    for p in candidates:
        lo = max(0, p - half_win)
        hi = min(len(ecg), p + half_win + 1)
        if inverted:
            local = lo + np.argmin(ecg[lo:hi])
        else:
            local = lo + np.argmax(ecg[lo:hi])

        # Высота относительно локального базиса
        win_bl = int(1.0 * fs)
        bl_s = max(0, local - win_bl)
        bl_e = min(len(ecg), local + win_bl)
        baseline = np.median(ecg[bl_s:bl_e])
        peak_indices.append(local)
        peak_heights.append(abs(ecg[local] - baseline))

    peak_indices = np.array(peak_indices)
    peak_heights = np.array(peak_heights)

    if len(peak_heights) < 2:
        return peak_indices

    # Отбрасываем мелкие пики (< 50% медианы высоты)
    keep = peak_heights >= 0.5 * np.median(peak_heights)
    return np.unique(peak_indices[keep])


# ═════════════════════════════════════════════════════════════════════════════
# 4. ДЕТЕКЦИЯ FOOT НА ПЛЕТИЗМОГРАММЕ
# ═════════════════════════════════════════════════════════════════════════════

def detect_feet_chest(signal, rpeaks, fs,
                      start_ms=FOOT_START_MS, end_ms=FOOT_END_MS):
    """
    Для каждого R-пика ищем foot (начало волны) в сигнале груди.
    Foot = минимум в окне [start_ms, 80% до систолического пика].
    """
    start_dt = int(start_ms * fs / 1000)
    end_dt   = int(end_ms   * fs / 1000)
    feet = []
    for r in rpeaks:
        lo, hi = r + start_dt, r + end_dt
        if hi >= len(signal):
            continue
        seg = signal[lo:hi]
        if len(seg) < 3:
            continue
        peak_rel    = np.argmax(seg)
        search_end  = max(1, int(peak_rel * 0.8))
        foot_rel    = np.argmin(seg[:search_end])
        feet.append(lo + foot_rel)
    return np.array(feet, dtype=int)


def detect_feet_arm(signal, feet_chest, fs,
                    ptt_min_ms=PTT_MIN_MS, ptt_max_ms=PTT_MAX_MS):
    """
    Для каждого foot груди ищем соответствующий foot на руке
    в окне PTT_MIN … PTT_MAX после foot груди.
    Возвращает массив индексов (-1 если пара не найдена).
    """
    start_dt = int(ptt_min_ms * fs / 1000)
    end_dt   = int(ptt_max_ms * fs / 1000)
    feet_arm = []
    for fc in feet_chest:
        lo, hi = fc + start_dt, fc + end_dt
        if hi >= len(signal) or lo < 0:
            feet_arm.append(-1)
            continue
        seg = signal[lo:hi]
        if len(seg) < 3:
            feet_arm.append(-1)
            continue
        feet_arm.append(lo + np.argmin(seg))
    return np.array(feet_arm, dtype=int)


# ═════════════════════════════════════════════════════════════════════════════
# 5. РАСЧЁТ PTT / PWV
# ═════════════════════════════════════════════════════════════════════════════

def compute_pwv(feet_chest, feet_arm, fs, distance_m):
    """
    Вычисляет PTT и PWV для валидных пар.
    Применяет MAD-фильтр (z < 3.5) для удаления выбросов.
    Возвращает: ptt_arr, pwv_arr, fc_valid, fa_valid (все без выбросов).
    """
    valid = feet_arm >= 0
    fc_v  = feet_chest[valid]
    fa_v  = feet_arm[valid]

    if len(fc_v) == 0:
        return np.array([]), np.array([]), fc_v, fa_v

    ptt = (fa_v - fc_v) / fs
    # Убираем физически невозможные значения (отрицательный PTT)
    positive = ptt > 0
    fc_v, fa_v, ptt = fc_v[positive], fa_v[positive], ptt[positive]

    if len(ptt) == 0:
        return np.array([]), np.array([]), fc_v, fa_v

    pwv = distance_m / ptt

    # MAD-фильтр выбросов
    med = np.median(ptt)
    mad = np.median(np.abs(ptt - med))
    if mad > 0:
        z    = np.abs(ptt - med) / (mad * 1.4826)
        keep = z < 3.5
    else:
        keep = np.ones(len(ptt), dtype=bool)

    return ptt[keep], pwv[keep], fc_v[keep], fa_v[keep]


# ═════════════════════════════════════════════════════════════════════════════
# 6. ВИЗУАЛИЗАЦИЯ
# ═════════════════════════════════════════════════════════════════════════════

def plot_results(time, ecg, rpeaks, chest, feet_chest,
                 wrist, feet_arm_valid, fc_final, fa_final, ptt_val, pwv_val):
    """
    Интерактивный трёхпанельный график.
    Rangeslider вынесен отдельно — zoom синхронизируется по всем панелям.
    """
    pwv_med = float(np.median(pwv_val)) if len(pwv_val) else float('nan')
    ptt_med = float(np.median(ptt_val)) if len(ptt_val) else float('nan')
    title   = (f'СРПВ = {pwv_med:.2f} м/с  |  '
               f'PTT = {ptt_med*1000:.1f} мс  |  '
               f'Пар: {len(ptt_val)}')

    # 4 строки: 3 сигнала + отдельный rangeslider-ряд
    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        row_heights=[0.32, 0.32, 0.32, 0.04],
        subplot_titles=('ЭКГ + R-пики', 'Грудь + foot', 'Рука + foot', ''),
        vertical_spacing=0.05)

    # ── ЭКГ ──────────────────────────────────────────────
    fig.add_trace(go.Scattergl(x=time, y=ecg,
        name='ЭКГ', line=dict(color='royalblue', width=0.8)), row=1, col=1)
    fig.add_trace(go.Scattergl(x=time[rpeaks], y=ecg[rpeaks],
        mode='markers', name='R-пики',
        marker=dict(color='red', size=7, symbol='x')), row=1, col=1)

    # ── Грудь ────────────────────────────────────────────
    fig.add_trace(go.Scattergl(x=time, y=chest,
        name='Грудь', line=dict(color='seagreen', width=0.8)), row=2, col=1)
    fig.add_trace(go.Scattergl(x=time[feet_chest], y=chest[feet_chest],
        mode='markers', name='Foot грудь',
        marker=dict(color='darkgreen', size=8, symbol='circle-open')), row=2, col=1)

    # PTT-стрелки (первые 5)
    for i in range(min(5, len(ptt_val))):
        fc, fa = fc_final[i], fa_final[i]
        if fa < 0:
            continue
        fig.add_trace(go.Scattergl(
            x=[time[fc], time[fa]],
            y=[chest[fc], wrist[fa]],
            mode='lines+markers',
            line=dict(color='black', dash='dot', width=1),
            marker=dict(size=5, color='black'),
            showlegend=(i == 0),
            name=f'PTT={ptt_val[i]*1000:.0f} мс'), row=2, col=1)

    # ── Рука ─────────────────────────────────────────────
    fig.add_trace(go.Scattergl(x=time, y=wrist,
        name='Рука', line=dict(color='mediumpurple', width=0.8)), row=3, col=1)
    if len(fa_final) > 0:
        fa_plot = fa_final[fa_final >= 0]
        fig.add_trace(go.Scattergl(x=time[fa_plot], y=wrist[fa_plot],
            mode='markers', name='Foot рука',
            marker=dict(color='darkviolet', size=8, symbol='circle-open')), row=3, col=1)

    # ── Rangeslider на row=4 (пустая ось) ───────────────
    # Дублируем ЭКГ тонкой линией как «карту» для ползунка
    fig.add_trace(go.Scattergl(
        x=time, y=ecg,
        line=dict(color='royalblue', width=0.5),
        showlegend=False), row=4, col=1)

    fig.update_layout(
        height=870,
        title=title,
        hovermode='x unified',
        xaxis4=dict(
            rangeslider=dict(visible=True, thickness=0.04),
            title='Время, с',
            type='linear',
        ),
    )
    fig.update_yaxes(title_text='z-score',    row=1, col=1)
    fig.update_yaxes(title_text='Амплитуда',  row=2, col=1)
    fig.update_yaxes(title_text='Амплитуда',  row=3, col=1)
    fig.update_yaxes(visible=False,           row=4, col=1)
    fig.show()


# ═════════════════════════════════════════════════════════════════════════════
# 7. ОСНОВНАЯ ФУНКЦИЯ
# ═════════════════════════════════════════════════════════════════════════════

def process_file(filepath=DATA_FILE, plot=True, verbose=True):
    """
    Полный пайплайн для одного CSV-файла.
    Возвращает dict с результатами.
    """
    def log(msg):
        if verbose:
            print(msg)

    log(f'\n{"="*55}')
    log(f'Файл: {filepath}')

    # ── Загрузка ──────────────────────────────────────────
    df = pd.read_csv(filepath, header=None)
    log(f'Строк: {len(df)}, столбцов: {df.shape[1]}')

    # Правильная временная ось: fs из прошивки
    time         = np.arange(len(df)) / FS
    ch_chest_raw = df.iloc[:, 1].values.astype(float)
    ch_wrist_raw = df.iloc[:, 2].values.astype(float)
    ecg_raw      = df.iloc[:, 4].values.astype(float)
    log(f'Длина записи: {time[-1]:.1f} с')

    # ── Предобработка ─────────────────────────────────────
    ecg_filt    = preprocess_ecg(ecg_raw, FS)
    ch_chest_lp = butter_bandpass_pleth(fix_zeros(ch_chest_raw))
    ch_wrist_lp = butter_bandpass_pleth(fix_zeros(ch_wrist_raw))

    # ── R-пики ────────────────────────────────────────────
    rpeaks = detect_rpeaks(ecg_filt, FS, MIN_RR_SEC, TKEO_FACTOR)
    log(f'R-пиков найдено: {len(rpeaks)}')
    if len(rpeaks) < 3:
        log('[!] Слишком мало R-пиков, обработка невозможна.')
        return None

    # ── Foot ──────────────────────────────────────────────
    feet_chest = detect_feet_chest(ch_chest_lp, rpeaks, FS,
                                   FOOT_START_MS, FOOT_END_MS)
    feet_arm   = detect_feet_arm(ch_wrist_lp, feet_chest, FS,
                                 PTT_MIN_MS, PTT_MAX_MS)
    log(f'Foot груди: {len(feet_chest)},  пар с рукой: {(feet_arm >= 0).sum()}')

    # ── PTT / PWV ─────────────────────────────────────────
    ptt_val, pwv_val, fc_final, fa_final = compute_pwv(
        feet_chest, feet_arm, FS, DISTANCE_M)

    if len(ptt_val) == 0:
        log('[!] Валидных PTT не найдено.')
        return None

    pwv_med = float(np.median(pwv_val))
    ptt_med = float(np.median(ptt_val))

    log(f'\nРезультаты:')
    log(f'  Валидных пар (после MAD-фильтра): {len(ptt_val)}')
    log(f'  PTT (медиана):  {ptt_med*1000:.1f} мс')
    log(f'  PTT (std):      {np.std(ptt_val)*1000:.1f} мс')
    log(f'  PWV (медиана):  {pwv_med:.2f} м/с')
    log(f'  PWV (std):      {np.std(pwv_val):.2f} м/с')

    result = dict(
        file       = str(filepath),
        duration_s = round(float(time[-1]), 1),
        n_rpeaks   = int(len(rpeaks)),
        n_valid    = int(len(ptt_val)),
        ptt_ms     = round(ptt_med * 1000, 1),
        ptt_std_ms = round(float(np.std(ptt_val)) * 1000, 1),
        pwv_ms     = round(pwv_med, 2),
        pwv_std    = round(float(np.std(pwv_val)), 2),
    )

    if plot:
        plot_results(time, ecg_filt, rpeaks,
                     ch_chest_lp, feet_chest,
                     ch_wrist_lp, fa_final,
                     fc_final, fa_final, ptt_val, pwv_val)

    return result


# ─── ТОЧКА ВХОДА ─────────────────────────────────────────────────────────────
if __name__ == '__main__':
    res = process_file(DATA_FILE, plot=True)
    if res:
        print('\nИтог:', res)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# -*- coding: utf-8 -*-
"""
Пайплайн расчёта СРПВ (PWV) — одиночный файл
МК: MSP430i2040  |  fs = 488.28 Гц (SMCLK=2.048 МГц, OSR=256, AVG=16)

Каналы в CSV:
  col 0 — счётчик (не используется для времени)
  col 1 — плетизмограмма грудь (Hall 1)
  col 2 — плетизмограмма рука  (Hall 2)
  col 3 — Hall 3, мусор (игнорируем)
  col 4 — ЭКГ
"""

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, savgol_filter
from scipy.ndimage import binary_dilation
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── НАСТРОЙКИ ───────────────────────────────────────────────────────────────
DATA_FILE    = 'data4ch_0_4.csv'
DISTANCE_M   = 0.5        # расстояние грудь–запястье, м
FS           = 488.28     # частота дискретизации из прошивки МК

# Детекция R-пиков
TKEO_FACTOR  = 0.6
MIN_RR_SEC   = 0.4

# Поиск foot
FOOT_START_MS = 50
FOOT_END_MS   = 400

# PTT
PTT_MIN_MS   = 20
PTT_MAX_MS   = 200

# ── Маскирование артефактов ───────────────────────────────────────────────
# RMS-окно: 2 с — достаточно, чтобы поймать дыхательный паттерн целиком.
# Уменьши RMS_THRESH, если маска не захватывает дыхательные участки.
# Увеличь,  если маска съедает слишком много хороших данных.
RMS_WINDOW_MS  = 2000   # ширина окна локального RMS, мс
RMS_THRESH     = 1.5    # порог: median_rms + thresh × MAD_rms
SPIKE_Z        = 5.0    # дополнительный порог для одиночных выбросов
ARTIFACT_EXP_MS = 300   # расширение маски вокруг артефакта, мс


# ═════════════════════════════════════════════════════════════════════════════
# 1. ФИЛЬТРЫ
# ═════════════════════════════════════════════════════════════════════════════

def butter_bandpass(data, low, high, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, min(high / nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)


def butter_bandpass_pleth(data, low=0.5, high=10.0, fs=FS, order=3):
    """
    Полосовой фильтр для плетизмограмм: убирает DC-дрейф и медленные
    дыхательные колебания (< 0.5 Гц), оставляет пульсовые волны (0.5–10 Гц).
    Примечание: быстрые переходы при вдохе/выдохе (> 0.5 Гц) не убираются —
    они обрабатываются маской артефактов на основе RMS.
    """
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, min(high / nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)


# ═════════════════════════════════════════════════════════════════════════════
# 2. УТИЛИТЫ
# ═════════════════════════════════════════════════════════════════════════════

def fix_zeros(sig):
    """Замена нулевых отсчётов линейной интерполяцией."""
    sig = sig.copy().astype(float)
    zero_mask = (sig == 0)
    if not zero_mask.any():
        return sig
    good_idx = np.where(~zero_mask)[0]
    if len(good_idx) == 0:
        return sig
    sig[zero_mask] = np.interp(np.where(zero_mask)[0], good_idx, sig[good_idx])
    return sig


def sliding_rms(signal, window):
    """Быстрое скользящее RMS через np.convolve."""
    sq  = signal.astype(float) ** 2
    rms = np.sqrt(np.convolve(sq, np.ones(window) / window, mode='same'))
    return rms


def mask_artifacts(signal, fs=FS,
                   rms_window_ms=RMS_WINDOW_MS,
                   rms_thresh=RMS_THRESH,
                   spike_z=SPIKE_Z,
                   expand_ms=ARTIFACT_EXP_MS):
    """
    Двухуровневое маскирование артефактов.

    Уровень 1 — RMS-маска (дыхание и движения):
        Вычисляет локальное RMS в скользящем окне rms_window_ms.
        Высокий локальный RMS = дыхательный паттерн или движение.
        Порог = median_rms + rms_thresh × MAD_rms (робастный).

    Уровень 2 — Spike-маска (одиночные выбросы):
        Точки, выходящие за spike_z робастных сигма от медианы.

    Финальная маска расширяется на ±expand_ms (переходный процесс фильтра).
    Возвращает bool-массив: True = артефакт.
    """
    # ── Уровень 1: RMS-маска ──────────────────────────────────────────────
    win    = max(3, int(rms_window_ms * fs / 1000))
    rms    = sliding_rms(signal, win)
    med_r  = np.median(rms)
    mad_r  = np.median(np.abs(rms - med_r)) * 1.4826
    thresh_rms  = med_r + rms_thresh * max(mad_r, 1e-10)
    rms_mask    = rms > thresh_rms

    # ── Уровень 2: Spike-маска ────────────────────────────────────────────
    med_s  = np.median(signal)
    mad_s  = np.median(np.abs(signal - med_s)) * 1.4826
    spike_mask  = np.abs(signal - med_s) > spike_z * max(mad_s, 1e-10)

    combined = rms_mask | spike_mask

    # ── Расширение маски ──────────────────────────────────────────────────
    if combined.any():
        expand   = int(expand_ms * fs / 1000)
        combined = binary_dilation(combined, structure=np.ones(2 * expand + 1))

    return combined


# ═════════════════════════════════════════════════════════════════════════════
# 3. ЭКГ
# ═════════════════════════════════════════════════════════════════════════════

def preprocess_ecg(raw, fs):
    ecg = butter_bandpass(raw, 0.5, min(40.0, fs * 0.45), fs)
    return (ecg - ecg.mean()) / (ecg.std() + 1e-8)


# ═════════════════════════════════════════════════════════════════════════════
# 4. R-ПИКИ (TKEO, робастный порог)
# ═════════════════════════════════════════════════════════════════════════════

def detect_rpeaks(ecg, fs, artifact_mask=None,
                  min_rr_sec=MIN_RR_SEC, tkeo_factor=TKEO_FACTOR):
    """
    Детекция R-пиков через TKEO.
    Артефактные участки обнуляются перед вычислением порога,
    чтобы крупные выбросы не завышали threshold.
    Порог = tkeo_factor × median(верхней половины TKEO) — устойчив к выбросам.
    """
    ecg_clean = ecg.copy()
    if artifact_mask is not None:
        ecg_clean[artifact_mask] = 0.0

    pos = np.max(ecg_clean) - np.median(ecg_clean)
    neg = np.median(ecg_clean) - np.min(ecg_clean)
    inverted  = neg > pos
    ecg_proc  = -ecg_clean if inverted else ecg_clean

    tkeo = ecg_proc[1:-1] ** 2 - ecg_proc[:-2] * ecg_proc[2:]
    tkeo = np.insert(tkeo, 0, 0.0)
    win  = max(3, int(0.05 * fs) | 1)
    tkeo_sm  = savgol_filter(tkeo, window_length=win, polyorder=2)
    tkeo_sm  = np.clip(tkeo_sm, 0, None)

    upper    = tkeo_sm[tkeo_sm >= np.median(tkeo_sm)]
    thresh   = tkeo_factor * np.median(upper)
    min_dist = int(min_rr_sec * fs)
    candidates, _ = find_peaks(tkeo_sm, height=thresh, distance=min_dist)

    half_win = max(1, int(0.03 * fs))
    indices, heights = [], []
    for p in candidates:
        lo  = max(0, p - half_win)
        hi  = min(len(ecg), p + half_win + 1)
        loc = lo + (np.argmin(ecg[lo:hi]) if inverted else np.argmax(ecg[lo:hi]))
        if artifact_mask is not None and artifact_mask[loc]:
            continue
        win_bl   = int(1.0 * fs)
        baseline = np.median(ecg[max(0, loc - win_bl):min(len(ecg), loc + win_bl)])
        indices.append(loc)
        heights.append(abs(ecg[loc] - baseline))

    indices = np.array(indices)
    heights = np.array(heights)
    if len(heights) < 2:
        return indices
    keep = heights >= 0.5 * np.median(heights)
    return np.unique(indices[keep])


# ═════════════════════════════════════════════════════════════════════════════
# 5. ДЕТЕКЦИЯ FOOT
# ═════════════════════════════════════════════════════════════════════════════

def detect_feet_chest(signal, rpeaks, fs, artifact_mask=None,
                      start_ms=FOOT_START_MS, end_ms=FOOT_END_MS):
    """
    Foot = минимум перед систолическим пиком в окне [start_ms … end_ms].
    Окна, перекрывающиеся с артефактной маской, пропускаются.
    """
    start_dt = int(start_ms * fs / 1000)
    end_dt   = int(end_ms   * fs / 1000)
    feet = []
    for r in rpeaks:
        lo, hi = r + start_dt, r + end_dt
        if hi >= len(signal):
            continue
        if artifact_mask is not None and artifact_mask[lo:hi].any():
            continue
        seg = signal[lo:hi]
        if len(seg) < 3:
            continue
        peak_rel   = np.argmax(seg)
        search_end = max(1, int(peak_rel * 0.8))
        feet.append(lo + np.argmin(seg[:search_end]))
    return np.array(feet, dtype=int)


def detect_feet_arm(signal, feet_chest, fs, artifact_mask=None,
                    ptt_min_ms=PTT_MIN_MS, ptt_max_ms=PTT_MAX_MS):
    """
    Для каждого foot груди ищет foot на руке в окне PTT_MIN … PTT_MAX.
    Возвращает -1 для пар, попавших в артефактную зону.
    """
    start_dt = int(ptt_min_ms * fs / 1000)
    end_dt   = int(ptt_max_ms * fs / 1000)
    feet_arm = []
    for fc in feet_chest:
        lo, hi = fc + start_dt, fc + end_dt
        if hi >= len(signal) or lo < 0:
            feet_arm.append(-1)
            continue
        if artifact_mask is not None and artifact_mask[lo:hi].any():
            feet_arm.append(-1)
            continue
        seg = signal[lo:hi]
        feet_arm.append(lo + np.argmin(seg))
    return np.array(feet_arm, dtype=int)


# ═════════════════════════════════════════════════════════════════════════════
# 6. PTT / PWV
# ═════════════════════════════════════════════════════════════════════════════

def compute_pwv(feet_chest, feet_arm, fs, distance_m):
    """
    PTT = (idx_arm − idx_chest) / fs.
    PWV = distance / PTT.
    Фильтрация: отрицательный PTT → выброс; MAD-фильтр z < 3.5.
    (Амплитудная проверка убрана — маска артефактов уже исключила
    биты в дыхательных сегментах.)
    """
    valid = feet_arm >= 0
    fc_v  = feet_chest[valid]
    fa_v  = feet_arm[valid]

    if len(fc_v) == 0:
        return np.array([]), np.array([]), fc_v, fa_v

    ptt      = (fa_v - fc_v) / fs
    positive = ptt > 0
    fc_v, fa_v, ptt = fc_v[positive], fa_v[positive], ptt[positive]

    if len(ptt) == 0:
        return np.array([]), np.array([]), fc_v, fa_v

    pwv = distance_m / ptt

    med  = np.median(ptt)
    mad  = np.median(np.abs(ptt - med)) * 1.4826
    keep = np.abs(ptt - med) / max(mad, 1e-9) < 3.5

    return ptt[keep], pwv[keep], fc_v[keep], fa_v[keep]


# ═════════════════════════════════════════════════════════════════════════════
# 7. ВИЗУАЛИЗАЦИЯ
# ═════════════════════════════════════════════════════════════════════════════

def plot_results(time, ecg, rpeaks, chest, feet_chest,
                 wrist, fc_final, fa_final, ptt_val, pwv_val,
                 artifact_mask=None):

    pwv_med = float(np.median(pwv_val)) if len(pwv_val) else float('nan')
    ptt_med = float(np.median(ptt_val)) if len(ptt_val) else float('nan')
    title   = (f'СРПВ = {pwv_med:.2f} м/с  |  '
               f'PTT = {ptt_med*1000:.1f} мс  |  '
               f'Пар: {len(ptt_val)}')

    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True,
        row_heights=[0.32, 0.32, 0.32, 0.04],
        subplot_titles=('ЭКГ + R-пики', 'Грудь + foot', 'Рука + foot', ''),
        vertical_spacing=0.05)

    # Артефактные зоны (красная заливка)
    if artifact_mask is not None and artifact_mask.any():
        diff   = np.diff(artifact_mask.astype(int))
        starts = list(np.where(diff ==  1)[0])
        ends   = list(np.where(diff == -1)[0])
        if artifact_mask[0]:
            starts.insert(0, 0)
        if artifact_mask[-1]:
            ends.append(len(artifact_mask) - 1)
        for s, e in zip(starts, ends):
            for row in range(1, 4):
                fig.add_vrect(
                    x0=time[s], x1=time[e],
                    fillcolor='rgba(220, 80, 80, 0.18)',
                    line_width=0, row=row, col=1)

    # ЭКГ
    fig.add_trace(go.Scattergl(x=time, y=ecg,
        name='ЭКГ', line=dict(color='royalblue', width=0.8)), row=1, col=1)
    fig.add_trace(go.Scattergl(x=time[rpeaks], y=ecg[rpeaks],
        mode='markers', name='R-пики',
        marker=dict(color='red', size=7, symbol='x')), row=1, col=1)

    # Грудь
    fig.add_trace(go.Scattergl(x=time, y=chest,
        name='Грудь', line=dict(color='seagreen', width=0.8)), row=2, col=1)
    if len(feet_chest):
        fig.add_trace(go.Scattergl(x=time[feet_chest], y=chest[feet_chest],
            mode='markers', name='Foot грудь',
            marker=dict(color='darkgreen', size=8, symbol='circle-open')), row=2, col=1)

    # PTT-стрелки (первые 5)
    for i in range(min(5, len(ptt_val))):
        fc, fa = fc_final[i], fa_final[i]
        fig.add_trace(go.Scattergl(
            x=[time[fc], time[fa]], y=[chest[fc], wrist[fa]],
            mode='lines+markers',
            line=dict(color='black', dash='dot', width=1),
            marker=dict(size=5, color='black'),
            showlegend=(i == 0),
            name=f'PTT={ptt_val[i]*1000:.0f} мс'), row=2, col=1)

    # Рука
    fig.add_trace(go.Scattergl(x=time, y=wrist,
        name='Рука', line=dict(color='mediumpurple', width=0.8)), row=3, col=1)
    if len(fa_final):
        fa_plot = fa_final[fa_final >= 0]
        if len(fa_plot):
            fig.add_trace(go.Scattergl(x=time[fa_plot], y=wrist[fa_plot],
                mode='markers', name='Foot рука',
                marker=dict(color='darkviolet', size=8, symbol='circle-open')), row=3, col=1)

    # Rangeslider
    fig.add_trace(go.Scattergl(
        x=time, y=ecg, line=dict(color='royalblue', width=0.5),
        showlegend=False), row=4, col=1)

    fig.update_layout(
        height=920, title=title, hovermode='x unified',
        xaxis4=dict(
            rangeslider=dict(visible=True, thickness=0.04),
            title='Время, с', type='linear'))
    fig.update_yaxes(title_text='z-score',   row=1, col=1)
    fig.update_yaxes(title_text='Амплитуда', row=2, col=1)
    fig.update_yaxes(title_text='Амплитуда', row=3, col=1)
    fig.update_yaxes(visible=False,          row=4, col=1)
    fig.show()


# ═════════════════════════════════════════════════════════════════════════════
# 8. ОСНОВНАЯ ФУНКЦИЯ
# ═════════════════════════════════════════════════════════════════════════════

def process_file(filepath=DATA_FILE, plot=True, verbose=True):
    """
    Полный пайплайн для одного CSV-файла.
    Возвращает dict с результатами или None при ошибке.
    """
    def log(msg):
        if verbose:
            print(msg)

    log(f'\n{"="*55}')
    log(f'Файл: {filepath}')

    try:
        df = pd.read_csv(filepath, header=None)
    except Exception as e:
        log(f'[!] Ошибка чтения: {e}')
        return None

    log(f'Строк: {len(df)}, столбцов: {df.shape[1]}')
    if df.shape[1] < 5 or len(df) < int(FS * 5):
        log('[!] Слишком короткая запись или неверная структура.')
        return None

    time         = np.arange(len(df)) / FS
    log(f'Длина записи: {time[-1]:.1f} с')

    ch_chest_raw = df.iloc[:, 1].values.astype(float)
    ch_wrist_raw = df.iloc[:, 2].values.astype(float)
    ecg_raw      = df.iloc[:, 4].values.astype(float)

    # Предобработка
    ecg_filt    = preprocess_ecg(ecg_raw, FS)
    ch_chest_bp = butter_bandpass_pleth(fix_zeros(ch_chest_raw))
    ch_wrist_bp = butter_bandpass_pleth(fix_zeros(ch_wrist_raw))

    # Маски артефактов
    mask_chest   = mask_artifacts(ch_chest_bp)
    mask_wrist   = mask_artifacts(ch_wrist_bp)
    mask_ecg     = mask_artifacts(ecg_filt, rms_thresh=2.5, spike_z=5.0,
                                  expand_ms=200)
    combined_mask = mask_chest | mask_wrist | mask_ecg

    art_pct = 100.0 * combined_mask.mean()
    log(f'Артефактов (маска): {art_pct:.1f}% записи')

    # R-пики
    rpeaks = detect_rpeaks(ecg_filt, FS,
                           artifact_mask=combined_mask,
                           min_rr_sec=MIN_RR_SEC,
                           tkeo_factor=TKEO_FACTOR)
    log(f'R-пиков найдено: {len(rpeaks)}')
    if len(rpeaks) < 3:
        log('[!] Слишком мало R-пиков.')
        return None

    # Foot
    feet_chest = detect_feet_chest(ch_chest_bp, rpeaks, FS,
                                   artifact_mask=combined_mask)
    feet_arm   = detect_feet_arm(ch_wrist_bp, feet_chest, FS,
                                 artifact_mask=combined_mask)
    log(f'Foot груди: {len(feet_chest)},  пар с рукой: {(feet_arm >= 0).sum()}')

    # PTT / PWV
    ptt_val, pwv_val, fc_final, fa_final = compute_pwv(
        feet_chest, feet_arm, FS, DISTANCE_M)

    if len(ptt_val) == 0:
        log('[!] Валидных PTT не найдено.')
        return None

    pwv_med = float(np.median(pwv_val))
    ptt_med = float(np.median(ptt_val))

    log(f'\nРезультаты:')
    log(f'  Валидных пар: {len(ptt_val)}')
    log(f'  PTT медиана:  {ptt_med*1000:.1f} мс  |  std: {np.std(ptt_val)*1000:.1f} мс')
    log(f'  PWV медиана:  {pwv_med:.2f} м/с  |  std: {np.std(pwv_val):.2f} м/с')

    result = dict(
        file         = str(filepath),
        duration_s   = round(float(time[-1]), 1),
        artifact_pct = round(art_pct, 1),
        n_rpeaks     = int(len(rpeaks)),
        n_valid      = int(len(ptt_val)),
        ptt_ms       = round(ptt_med * 1000, 1),
        ptt_std_ms   = round(float(np.std(ptt_val)) * 1000, 1),
        pwv_ms       = round(pwv_med, 2),
        pwv_std      = round(float(np.std(pwv_val)), 2),
    )

    if plot:
        plot_results(time, ecg_filt, rpeaks,
                     ch_chest_bp, feet_chest,
                     ch_wrist_bp, fc_final, fa_final,
                     ptt_val, pwv_val,
                     artifact_mask=combined_mask)

    return result


# ─── ТОЧКА ВХОДА ─────────────────────────────────────────────────────────────
if __name__ == '__main__':
    res = process_file(DATA_FILE, plot=True)
    if res:
        print('\nИтог:', res)


Файл: data4ch_0_4.csv
Строк: 20838, столбцов: 5
Длина записи: 42.7 с
Артефактов (маска): 100.0% записи
R-пиков найдено: 0
[!] Слишком мало R-пиков.


In [2]:
# -*- coding: utf-8 -*-
"""
Пайплайн расчёта СРПВ (PWV) — одиночный файл
МК: MSP430i2040  |  fs = 488.28 Гц (SMCLK=2.048 МГц, OSR=256, AVG=16)

Каналы в CSV:
  col 0 — счётчик (не используется для времени)
  col 1 — плетизмограмма грудь (Hall 1)
  col 2 — плетизмограмма рука  (Hall 2)
  col 3 — Hall 3, мусор (игнорируем)
  col 4 — ЭКГ
"""

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, find_peaks, savgol_filter
from scipy.ndimage import binary_dilation
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ─── НАСТРОЙКИ ───────────────────────────────────────────────────────────────
DATA_FILE     = 'data4ch_0_7.csv'
DISTANCE_M    = 0.5       # расстояние грудь–запястье, м
FS            = 488.28    # частота дискретизации из прошивки МК

# Детекция R-пиков
TKEO_FACTOR   = 0.6
MIN_RR_SEC    = 0.4

# Поиск foot
FOOT_START_MS = 50
FOOT_END_MS   = 400

# PTT
PTT_MIN_MS    = 20
PTT_MAX_MS    = 200

# Маскирование артефактов плетизмограмм
# Дыхательные движения создают резкие выбросы в bandpass-сигнале.
# ARTIFACT_Z  — порог в единицах MAD-σ. Пульсовые волны ~2–4σ, дыхательные
#               выбросы ~10–30σ. Значение 6.0 оставляет пульс нетронутым.
# ARTIFACT_EXP_MS — расширение маски (переходный процесс фильтра), мс.
ARTIFACT_Z       = 6.0
ARTIFACT_EXP_MS  = 150


# ═════════════════════════════════════════════════════════════════════════════
# 1. ФИЛЬТРЫ
# ═════════════════════════════════════════════════════════════════════════════

def butter_bandpass(data, low, high, fs, order=3):
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, min(high / nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)


def butter_bandpass_pleth(data, low=0.5, high=10.0, fs=FS, order=3):
    """
    Полосовой фильтр для плетизмограмм.
    Нижний порог 0.5 Гц убирает DC-дрейф и медленные дыхательные
    колебания (< 0.5 Гц), оставляя пульсовые волны (0.5–10 Гц).
    Быстрые переходы при вдохе/выдохе остаются — они детектируются
    и исключаются маской артефактов (spike-based).
    """
    nyq = 0.5 * fs
    b, a = butter(order, [low / nyq, min(high / nyq, 0.99)], btype='band')
    return filtfilt(b, a, data)


# ═════════════════════════════════════════════════════════════════════════════
# 2. УТИЛИТЫ
# ═════════════════════════════════════════════════════════════════════════════

def fix_zeros(sig):
    """Замена нулевых отсчётов линейной интерполяцией."""
    sig = sig.copy().astype(float)
    zero_mask = (sig == 0)
    if not zero_mask.any():
        return sig
    good_idx = np.where(~zero_mask)[0]
    if len(good_idx) == 0:
        return sig
    sig[zero_mask] = np.interp(np.where(zero_mask)[0], good_idx, sig[good_idx])
    return sig


def mask_artifacts(signal, fs=FS,
                   z_thresh=ARTIFACT_Z,
                   expand_ms=ARTIFACT_EXP_MS):
    """
    Маскирование дыхательных артефактов в плетизмограмме (только spike-based).

    После bandpass-фильтрации (0.5–10 Гц) медленные дыхательные колебания
    убраны. Остаются резкие выбросы в момент вдоха/выдоха (~10–30σ) —
    именно их и ловит этот детектор.

    Порог считается через MAD (не std), чтобы сами выбросы не завышали σ.
    Маска расширяется на ±expand_ms для покрытия переходного процесса фильтра.

    НЕ применяется к ЭКГ: R-пики (~5–6σ) были бы ошибочно помечены
    как артефакты.
    """
    med = np.median(signal)
    mad = np.median(np.abs(signal - med)) * 1.4826
    if mad < 1e-10:
        return np.zeros(len(signal), dtype=bool)

    mask = np.abs(signal - med) > z_thresh * mad

    if mask.any():
        expand = int(expand_ms * fs / 1000)
        mask   = binary_dilation(mask, structure=np.ones(2 * expand + 1))

    return mask


# ═════════════════════════════════════════════════════════════════════════════
# 3. ЭКГ
# ═════════════════════════════════════════════════════════════════════════════

def preprocess_ecg(raw, fs):
    ecg = butter_bandpass(raw, 0.5, min(10.0, fs * 0.45), fs)
    return (ecg - ecg.mean()) / (ecg.std() + 1e-8)


# ═════════════════════════════════════════════════════════════════════════════
# 4. R-ПИКИ (TKEO, робастный порог)
# ═════════════════════════════════════════════════════════════════════════════

def detect_rpeaks(ecg, fs,
                  min_rr_sec=MIN_RR_SEC, tkeo_factor=TKEO_FACTOR):
    """
    Детекция R-пиков через TKEO.
    Маска артефактов здесь НЕ применяется: R-пики детектируются по всей
    записи, артефактные участки исключаются позже — при поиске foot.
    Порог = tkeo_factor × median(верхней половины TKEO): устойчив к выбросам.
    """
    pos = np.max(ecg) - np.median(ecg)
    neg = np.median(ecg) - np.min(ecg)
    inverted  = neg > pos
    ecg_proc  = -ecg if inverted else ecg

    tkeo = ecg_proc[1:-1] ** 2 - ecg_proc[:-2] * ecg_proc[2:]
    tkeo = np.insert(tkeo, 0, 0.0)
    win  = max(3, int(0.05 * fs) | 1)
    tkeo_sm = savgol_filter(tkeo, window_length=win, polyorder=2)
    tkeo_sm = np.clip(tkeo_sm, 0, None)

    upper   = tkeo_sm[tkeo_sm >= np.median(tkeo_sm)]
    thresh  = tkeo_factor * np.median(upper)
    min_dist = int(min_rr_sec * fs)
    candidates, _ = find_peaks(tkeo_sm, height=thresh, distance=min_dist)

    half_win = max(1, int(0.03 * fs))
    indices, heights = [], []
    for p in candidates:
        lo  = max(0, p - half_win)
        hi  = min(len(ecg), p + half_win + 1)
        loc = lo + (np.argmin(ecg[lo:hi]) if inverted else np.argmax(ecg[lo:hi]))
        win_bl   = int(1.0 * fs)
        baseline = np.median(ecg[max(0, loc - win_bl):min(len(ecg), loc + win_bl)])
        indices.append(loc)
        heights.append(abs(ecg[loc] - baseline))

    indices = np.array(indices)
    heights = np.array(heights)
    if len(heights) < 2:
        return indices
    keep = heights >= 0.5 * np.median(heights)
    return np.unique(indices[keep])


# ═════════════════════════════════════════════════════════════════════════════
# 5. ДЕТЕКЦИЯ FOOT
# ═════════════════════════════════════════════════════════════════════════════

def detect_feet_chest(signal, rpeaks, fs, artifact_mask=None,
                      start_ms=FOOT_START_MS, end_ms=FOOT_END_MS):
    """
    Foot = минимум перед систолическим пиком в окне [start_ms … end_ms].
    Окна, перекрывающиеся с маской артефактов, пропускаются.
    """
    start_dt = int(start_ms * fs / 1000)
    end_dt   = int(end_ms   * fs / 1000)
    feet = []
    for r in rpeaks:
        lo, hi = r + start_dt, r + end_dt
        if hi >= len(signal):
            continue
        if artifact_mask is not None and artifact_mask[lo:hi].any():
            continue
        seg = signal[lo:hi]
        if len(seg) < 3:
            continue
        peak_rel   = np.argmax(seg)
        search_end = max(1, int(peak_rel * 0.8))
        feet.append(lo + np.argmin(seg[:search_end]))
    return np.array(feet, dtype=int)


def detect_feet_arm(signal, feet_chest, fs, artifact_mask=None,
                    ptt_min_ms=PTT_MIN_MS, ptt_max_ms=PTT_MAX_MS):
    """
    Для каждого foot груди ищет foot на руке в окне PTT_MIN … PTT_MAX.
    Возвращает -1 для пар в артефактной зоне.
    """
    start_dt = int(ptt_min_ms * fs / 1000)
    end_dt   = int(ptt_max_ms * fs / 1000)
    feet_arm = []
    for fc in feet_chest:
        lo, hi = fc + start_dt, fc + end_dt
        if hi >= len(signal) or lo < 0:
            feet_arm.append(-1)
            continue
        if artifact_mask is not None and artifact_mask[lo:hi].any():
            feet_arm.append(-1)
            continue
        seg = signal[lo:hi]
        feet_arm.append(lo + np.argmin(seg))
    return np.array(feet_arm, dtype=int)


# ═════════════════════════════════════════════════════════════════════════════
# 6. PTT / PWV
# ═════════════════════════════════════════════════════════════════════════════

def compute_pwv(feet_chest, feet_arm, fs, distance_m):
    """
    PTT = (idx_arm − idx_chest) / fs,  PWV = distance / PTT.
    Фильтрация: отрицательный PTT → выброс; MAD-фильтр (z < 3.5).
    """
    valid = feet_arm >= 0
    fc_v  = feet_chest[valid]
    fa_v  = feet_arm[valid]
    if len(fc_v) == 0:
        return np.array([]), np.array([]), fc_v, fa_v

    ptt = (fa_v - fc_v) / fs
    pos = ptt > 0
    fc_v, fa_v, ptt = fc_v[pos], fa_v[pos], ptt[pos]
    if len(ptt) == 0:
        return np.array([]), np.array([]), fc_v, fa_v

    pwv = distance_m / ptt
    med = np.median(ptt)
    mad = np.median(np.abs(ptt - med)) * 1.4826
    keep = np.abs(ptt - med) / max(mad, 1e-9) < 3.5
    return ptt[keep], pwv[keep], fc_v[keep], fa_v[keep]


# ═════════════════════════════════════════════════════════════════════════════
# 7. ВИЗУАЛИЗАЦИЯ
# ═════════════════════════════════════════════════════════════════════════════

def plot_results(time, ecg, rpeaks, chest, feet_chest,
                 wrist, fc_final, fa_final, ptt_val, pwv_val,
                 artifact_mask=None):

    pwv_med = float(np.median(pwv_val)) if len(pwv_val) else float('nan')
    ptt_med = float(np.median(ptt_val)) if len(ptt_val) else float('nan')
    title   = (f'СРПВ = {pwv_med:.2f} м/с  |  '
               f'PTT = {ptt_med*1000:.1f} мс  |  '
               f'Пар: {len(ptt_val)}')

    fig = make_subplots(
        rows=4, cols=1, shared_xaxes=True,
        row_heights=[0.32, 0.32, 0.32, 0.04],
        subplot_titles=('ЭКГ + R-пики', 'Грудь + foot', 'Рука + foot', ''),
        vertical_spacing=0.05)

    # Артефактные зоны (красная заливка)
    if artifact_mask is not None and artifact_mask.any():
        diff   = np.diff(artifact_mask.astype(int))
        starts = list(np.where(diff ==  1)[0])
        ends   = list(np.where(diff == -1)[0])
        if artifact_mask[0]:
            starts.insert(0, 0)
        if artifact_mask[-1]:
            ends.append(len(artifact_mask) - 1)
        for s, e in zip(starts, ends):
            for row in range(1, 4):
                fig.add_vrect(x0=time[s], x1=time[e],
                              fillcolor='rgba(220,80,80,0.18)',
                              line_width=0, row=row, col=1)

    # ЭКГ
    fig.add_trace(go.Scattergl(x=time, y=ecg, name='ЭКГ',
        line=dict(color='royalblue', width=0.8)), row=1, col=1)
    fig.add_trace(go.Scattergl(x=time[rpeaks], y=ecg[rpeaks],
        mode='markers', name='R-пики',
        marker=dict(color='red', size=7, symbol='x')), row=1, col=1)

    # Грудь
    fig.add_trace(go.Scattergl(x=time, y=chest, name='Грудь',
        line=dict(color='seagreen', width=0.8)), row=2, col=1)
    if len(feet_chest):
        fig.add_trace(go.Scattergl(x=time[feet_chest], y=chest[feet_chest],
            mode='markers', name='Foot грудь',
            marker=dict(color='darkgreen', size=8, symbol='circle-open')), row=2, col=1)

    # PTT-стрелки (первые 5)
    for i in range(min(5, len(ptt_val))):
        fc, fa = fc_final[i], fa_final[i]
        fig.add_trace(go.Scattergl(
            x=[time[fc], time[fa]], y=[chest[fc], wrist[fa]],
            mode='lines+markers',
            line=dict(color='black', dash='dot', width=1),
            marker=dict(size=5, color='black'),
            showlegend=(i == 0),
            name=f'PTT={ptt_val[i]*1000:.0f} мс'), row=2, col=1)

    # Рука
    fig.add_trace(go.Scattergl(x=time, y=wrist, name='Рука',
        line=dict(color='mediumpurple', width=0.8)), row=3, col=1)
    if len(fa_final):
        fa_plot = fa_final[fa_final >= 0]
        if len(fa_plot):
            fig.add_trace(go.Scattergl(x=time[fa_plot], y=wrist[fa_plot],
                mode='markers', name='Foot рука',
                marker=dict(color='darkviolet', size=8, symbol='circle-open')), row=3, col=1)

    # Rangeslider
    fig.add_trace(go.Scattergl(x=time, y=ecg,
        line=dict(color='royalblue', width=0.5), showlegend=False), row=4, col=1)

    fig.update_layout(height=920, title=title, hovermode='x unified',
        xaxis4=dict(rangeslider=dict(visible=True, thickness=0.04),
                    title='Время, с', type='linear'))
    fig.update_yaxes(title_text='z-score',   row=1, col=1)
    fig.update_yaxes(title_text='Амплитуда', row=2, col=1)
    fig.update_yaxes(title_text='Амплитуда', row=3, col=1)
    fig.update_yaxes(visible=False,          row=4, col=1)
    fig.show()


# ═════════════════════════════════════════════════════════════════════════════
# 8. ОСНОВНАЯ ФУНКЦИЯ
# ═════════════════════════════════════════════════════════════════════════════

def process_file(filepath=DATA_FILE, plot=True, verbose=True):
    """Полный пайплайн для одного CSV-файла."""
    def log(msg):
        if verbose:
            print(msg)

    log(f'\n{"="*55}')
    log(f'Файл: {filepath}')

    try:
        df = pd.read_csv(filepath, header=None)
    except Exception as e:
        log(f'[!] Ошибка чтения: {e}')
        return None

    log(f'Строк: {len(df)}, столбцов: {df.shape[1]}')
    if df.shape[1] < 5 or len(df) < int(FS * 5):
        log('[!] Слишком короткая запись или неверная структура.')
        return None

    time         = np.arange(len(df)) / FS
    log(f'Длина записи: {time[-1]:.1f} с')

    ch_chest_raw = df.iloc[:, 1].values.astype(float)
    ch_wrist_raw = df.iloc[:, 2].values.astype(float)
    ecg_raw      = df.iloc[:, 4].values.astype(float)

    # Предобработка сигналов
    ecg_filt    = preprocess_ecg(ecg_raw, FS)
    ch_chest_bp = butter_bandpass_pleth(fix_zeros(ch_chest_raw))
    ch_wrist_bp = butter_bandpass_pleth(fix_zeros(ch_wrist_raw))

    # Маска артефактов — только по плетизмограммам, ЭКГ не трогаем
    mask_chest    = mask_artifacts(ch_chest_bp)
    mask_wrist    = mask_artifacts(ch_wrist_bp)
    combined_mask = mask_chest | mask_wrist

    art_pct = 100.0 * combined_mask.mean()
    log(f'Артефактов (маска): {art_pct:.1f}% записи')

    # R-пики (без маски — находим по всей записи)
    rpeaks = detect_rpeaks(ecg_filt, FS,
                           min_rr_sec=MIN_RR_SEC,
                           tkeo_factor=TKEO_FACTOR)
    log(f'R-пиков найдено: {len(rpeaks)}')
    if len(rpeaks) < 3:
        log('[!] Слишком мало R-пиков.')
        return None

    # Foot (маска исключает биты в артефактных зонах)
    feet_chest = detect_feet_chest(ch_chest_bp, rpeaks, FS,
                                   artifact_mask=combined_mask)
    feet_arm   = detect_feet_arm(ch_wrist_bp, feet_chest, FS,
                                 artifact_mask=combined_mask)
    log(f'Foot груди: {len(feet_chest)},  пар с рукой: {(feet_arm >= 0).sum()}')

    # PTT / PWV
    ptt_val, pwv_val, fc_final, fa_final = compute_pwv(
        feet_chest, feet_arm, FS, DISTANCE_M)

    if len(ptt_val) == 0:
        log('[!] Валидных PTT не найдено.')
        return None

    pwv_med = float(np.median(pwv_val))
    ptt_med = float(np.median(ptt_val))

    log(f'\nРезультаты:')
    log(f'  Валидных пар: {len(ptt_val)}')
    log(f'  PTT медиана:  {ptt_med*1000:.1f} мс  |  std: {np.std(ptt_val)*1000:.1f} мс')
    log(f'  PWV медиана:  {pwv_med:.2f} м/с  |  std: {np.std(pwv_val):.2f} м/с')

    result = dict(
        file         = str(filepath),
        duration_s   = round(float(time[-1]), 1),
        artifact_pct = round(art_pct, 1),
        n_rpeaks     = int(len(rpeaks)),
        n_valid      = int(len(ptt_val)),
        ptt_ms       = round(ptt_med * 1000, 1),
        ptt_std_ms   = round(float(np.std(ptt_val)) * 1000, 1),
        pwv_ms       = round(pwv_med, 2),
        pwv_std      = round(float(np.std(pwv_val)), 2),
    )

    if plot:
        plot_results(time, ecg_filt, rpeaks,
                     ch_chest_bp, feet_chest,
                     ch_wrist_bp, fc_final, fa_final,
                     ptt_val, pwv_val,
                     artifact_mask=combined_mask)
    return result


# ─── ТОЧКА ВХОДА ─────────────────────────────────────────────────────────────
if __name__ == '__main__':
    res = process_file(DATA_FILE, plot=True)
    if res:
        print('\nИтог:', res)




---



---



---



---


---



---



In [ ]:
# Ячейка 1 — подключение диска (если ещё не подключён)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Ячейка 2 — установка зависимостей (если нужно)
!pip install scipy plotly -q

In [ ]:
# Ячейка с запуском батча — полная версия
import sys

DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/WORK/DATA'
CONFIG   = f'{DATA_DIR}/config.json'   # ← эта строка нужна обязательно
sys.path.insert(0, DATA_DIR)

from batch_pwv import run_batch

df_signals, df_patients, df_features, df_good = run_batch(
    config_path = CONFIG,
    data_dir    = DATA_DIR,
    plot        = False,
)


############################################################
  ПАЦИЕНТ P01   dist=0.57 м   tkeo=0.8
  Файлов: 14
############################################################

Файл: /content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_0_1.csv
Строк: 24997, столбцов: 5
Длина записи: 51.2 с
Артефактов (маска): 26.0%
R-пиков найдено: 79
Foot груди: 49,  пар с рукой: 49

Результаты:
  Валидных пар: 49
  PTT медиана: 65.5 мс  std: 69.1 мс
  PWV медиана: 8.70 м/с  std: 8.98 м/с
  HR: 97.5 уд/мин  RMSSD: 154.5 мс  SDNN: 162.9 мс

Файл: /content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_0_2.csv
Строк: 31749, столбцов: 5
Длина записи: 65.0 с
Артефактов (маска): 30.9%
R-пиков найдено: 99
Foot груди: 65,  пар с рукой: 65

Результаты:
  Валидных пар: 65
  PTT медиана: 108.5 мс  std: 78.9 мс
  PWV медиана: 5.25 м/с  std: 10.89 м/с
  HR: 97.5 уд/мин  RMSSD: 152.6 мс  SDNN: 150.3 мс

Файл: /content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_0_3.csv
Строк: 32140, столбцов: 5
Длина запис

In [ ]:
# Ячейка 4 — просмотр результатов
display(df_patients)
display(df_signals)

,patient_id,n_signals,n_signals_ok,pwv_mean,pwv_std_signals,pwv_min,pwv_max,ptt_mean_ms,ptt_std_ms,mean_hr,sdnn_ms,rmssd_ms,pnn50,avg_artifact_pct,avg_valid_ratio
0,P01,13,13,5.54,6.46,3.20,27.83,115.8,41.2,99.2,140.3,143.8,33.5,25.4,0.645
1,P02,6,6,17.70,9.73,6.05,25.39,44.5,30.9,136.9,58.5,62.9,3.4,12.4,0.586
2,P03,7,7,4.68,5.78,1.91,18.55,108.7,51.8,124.8,86.6,81.3,12.3,31.1,0.552


,file,duration_s,artifact_pct,n_rpeaks,n_valid,ptt_ms,ptt_std_ms,pwv_ms,pwv_std,mean_hr,sdnn_ms,rmssd_ms,pnn50,patient_id,signal_idx
0,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,51.2,26.0,79,49,65.5,69.1,8.70,8.98,97.5,162.9,154.5,36.7,P01,1
1,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,65.0,30.9,99,65,108.5,78.9,5.25,10.89,97.5,150.3,152.6,28.4,P01,2
2,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,65.8,28.3,101,66,146.4,79.1,3.90,10.70,98.3,182.4,206.7,52.5,P01,3
3,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,42.7,24.5,67,41,178.2,50.2,3.20,2.73,105.3,129.0,151.3,31.6,P01,4
4,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,58.9,29.9,96,65,135.2,70.9,4.22,9.79,106.9,113.8,120.9,34.9,P01,5
5,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,55.7,30.1,86,58,104.4,40.2,5.46,6.52,97.8,142.8,130.1,14.8,P01,6
6,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,67.5,24.0,111,64,98.3,19.8,5.80,1.58,106.4,131.6,146.5,42.9,P01,7
7,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,58.9,25.9,94,63,77.8,62.3,7.32,8.23,102.0,120.3,122.1,32.8,P01,8
8,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,73.9,20.2,118,88,106.5,57.5,5.35,7.62,102.6,113.5,118.5,31.7,P01,9
9,/content/drive/MyDrive/Colab Notebooks/WORK/DA...,51.2,21.4,81,60,146.4,63.8,3.89,8.58,99.1,146.7,130.9,29.8,P01,10


In [ ]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/Colab Notebooks/WORK/DATA')

from pwv_single import process_file

# Один файл с графиком
result = process_file(
    filepath      = '/content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_1_5.csv',
    distance_m    = 0.52,    # параметры P02 из config.json
    tkeo_factor   = 0.6,
    plot          = True,
)

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
from pwv_single import process_file

result = process_file(
    filepath      = '/content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_4_1.csv',
    distance_m    = 0.38,
    tkeo_factor   = 0.4,
    plot          = True,
)

Output hidden; open in https://colab.research.google.com to view.



---



---



---



In [ ]:
# Ячейка 1 — батч + визуализация
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/WORK/DATA'
CONFIG   = f'{DATA_DIR}/config.json'
import sys; sys.path.insert(0, DATA_DIR)

from batch_pwv import run_batch, plot_patient_comparison

df_signals, df_patients, df_features, df_good = run_batch(
    config_path=CONFIG, data_dir=DATA_DIR, plot=False)

# Дашборд сравнения пациентов
patient_meta = {
    'P01': {'age': 26, 'notes': 'аритмия, лишний вес'},
    'P02': {'age': 40, 'notes': 'норма'},
    'P03': {'age': 55, 'notes': 'норма, жен.'},
}
plot_patient_comparison(df_features, patient_meta)


############################################################
  ПАЦИЕНТ P01   dist=0.57 м   tkeo=0.8
  Файлов: 14
############################################################

Файл: /content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_0_1.csv
Строк: 24997, столбцов: 5
Длина записи: 51.2 с
Артефактов (маска): 26.0%
R-пиков найдено: 79
Foot груди: 49,  пар с рукой: 49

Результаты:
  Валидных пар: 49
  PTT медиана: 65.5 мс  std: 69.1 мс
  PWV медиана: 8.70 м/с  std: 8.98 м/с
  HR: 97.5 уд/мин  RMSSD: 154.5 мс  SDNN: 162.9 мс

Файл: /content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_0_2.csv
Строк: 31749, столбцов: 5
Длина записи: 65.0 с
Артефактов (маска): 30.9%
R-пиков найдено: 99
Foot груди: 65,  пар с рукой: 65

Результаты:
  Валидных пар: 65
  PTT медиана: 108.5 мс  std: 78.9 мс
  PWV медиана: 5.25 м/с  std: 10.89 м/с
  HR: 97.5 уд/мин  RMSSD: 152.6 мс  SDNN: 150.3 мс

Файл: /content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_0_3.csv
Строк: 32140, столбцов: 5
Длина запис

In [ ]:
QUALITY_CSV = f'{DATA_DIR}/processed_data/features_quality_ok.csv'
from pwv_cnn import build_dataset, train_cnn, evaluate_cnn, plot_training_history

# Шаг 1 — считаем биты
X, y, meta = build_dataset(CONFIG, DATA_DIR, QUALITY_CSV)
print(f'Форма X: {X.shape}')  # ожидаем (~400, 293, 3)
print(f'PTT мин/макс: {y.min():.1f} / {y.max():.1f} мс')

quality_csv не задан — берём все сигналы
  Извлечение битов: data4ch_0_1.csv ... 49 битов
  Извлечение битов: data4ch_0_2.csv ... 65 битов
  Извлечение битов: data4ch_0_3.csv ... 66 битов
  Извлечение битов: data4ch_0_4.csv ... 41 битов
  Извлечение битов: data4ch_0_5.csv ... 65 битов
  Извлечение битов: data4ch_0_6.csv ... 58 битов
  Извлечение битов: data4ch_0_7.csv ... 64 битов
  Извлечение битов: data4ch_0_8.csv ... 62 битов
  Извлечение битов: data4ch_0_9.csv ... 88 битов
  Извлечение битов: data4ch_0_10.csv ... 60 битов
  Извлечение битов: data4ch_0_11.csv ... 12 битов
  Извлечение битов: data4ch_0_12.csv ... 49 битов
  Извлечение битов: data4ch_0_13.csv ... 75 битов
  Извлечение битов: data4ch_0_14.csv ... 0 битов
  Извлечение битов: data4ch_1_1.csv ... 54 битов
  Извлечение битов: data4ch_1_2.csv ... 78 битов
  Извлечение битов: data4ch_1_3.csv ... 143 битов
  Извлечение битов: data4ch_1_4.csv ... 176 битов
  Извлечение битов: data4ch_1_5.csv ... 51 битов
  Извлечение битов: da

In [ ]:
# Фильтруем плохие PTT (> 25 мс — убираем зажатые 20.5 мс)
valid_mask = (y > 25) & (y <= 250)
X_clean    = X[valid_mask]
y_clean    = y[valid_mask]
meta_clean = [m for m, v in zip(meta, valid_mask) if v]
import numpy as np
print(f'Всего битов:          {len(y)}')
print(f'После фильтра PTT>25: {len(y_clean)}')
print(f'Форма X: {X_clean.shape}')
print(f'PTT медиана: {np.median(y_clean):.1f} мс  '
      f'[{y_clean.min():.1f} – {y_clean.max():.1f}]')

# Распределение по пациентам
import pandas as pd
df_meta = pd.DataFrame(meta_clean)
print('\nБитов по пациентам:')
print(df_meta['patient_id'].value_counts())

Всего битов:          1982
После фильтра PTT>25: 1286
Форма X: (1286, 292, 3)
PTT медиана: 121.9 мс  [26.6 – 198.7]

Битов по пациентам:
patient_id
P01    621
P03    517
P02    148
Name: count, dtype: int64


In [ ]:
model, history, X_test, y_test, meta_test = train_cnn(
    X_clean, y_clean, meta_clean,
    epochs=60,
    save_path=f'{DATA_DIR}/pwv_cnn.h5',
)

Стратегия: leave-one-patient-out, тест = P02
Обучение: 1138 битов  |  Тест: 148 битов


Model: "PWV_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ signal_input (InputLayer)       │ (None, 292, 3)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 292, 32)        │         3,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 292, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 146, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 146, 64)        │        38,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 146, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 73, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_6 (Conv1D)               │ (None, 73, 128)        │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 73, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 36, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_7 (Conv1D)               │ (None, 36, 64)         │        41,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ptt_ms (Dense)                  │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 164,801 (643.75 KB)

 Trainable params: 164,353 (642.00 KB)

...

31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 86ms/step - loss: 27.3364 - mae: 27.8075 - val_loss: 21.4899 - val_mae: 21.9636 - learning_rate: 0.0010
Epoch 20/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 86ms/step - loss: 28.0233 - mae: 28.4912 - val_loss: 27.7742 - val_mae: 28.2429 - learning_rate: 0.0010
Epoch 21/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - loss: 26.0758 - mae: 26.5451 - val_loss: 26.4565 - val_mae: 26.9256 - learning_rate: 0.0010
Epoch 22/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 90ms/step - loss: 25.4571 - mae: 25.9264 - val_loss: 27.3867 - val_mae: 27.8537 - learning_rate: 0.0010
Epoch 23/60
30/31 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 25.0872 - mae: 25.5569

31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 102ms/step - loss: 25.0227 - mae: 25.4920 - val_loss: 20.5882 - val_mae: 21.0533 - learning_rate: 0.0010
Epoch 24/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 93ms/step - loss: 25.5584 - mae: 26.0274 - val_loss: 24.4133 - val_mae: 24.8843 - learning_rate: 0.0010
Epoch 25/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 5s 166ms/step - loss: 24.5520 - mae: 25.0210 - val_loss: 25.3841 - val_mae: 25.8542 - learning_rate: 0.0010
Epoch 26/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - loss: 24.3386 - mae: 24.8055 - val_loss: 21.2213 - val_mae: 21.6858 - learning_rate: 0.0010
Epoch 27/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - loss: 24.5325 - mae: 24.9989 - val_loss: 21.7514 - val_mae: 22.2054 - learning_rate: 0.0010
Epoch 28/60
30/31 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - loss: 23.8064 - mae: 24.2755

31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 105ms/step - loss: 23.8868 - mae: 24.3545 - val_loss: 19.4303 - val_mae: 19.8956 - learning_rate: 0.0010
Epoch 29/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 117ms/step - loss: 23.8679 - mae: 24.3352 - val_loss: 20.1479 - val_mae: 20.6112 - learning_rate: 0.0010
Epoch 30/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 123ms/step - loss: 23.0797 - mae: 23.5456 - val_loss: 28.2600 - val_mae: 28.7250 - learning_rate: 0.0010
Epoch 31/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 84ms/step - loss: 22.7984 - mae: 23.2653 - val_loss: 21.7667 - val_mae: 22.2350 - learning_rate: 0.0010
Epoch 32/60
30/31 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - loss: 21.7737 - mae: 22.2385

31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - loss: 22.5469 - mae: 23.0096 - val_loss: 18.7647 - val_mae: 19.2216 - learning_rate: 0.0010
Epoch 33/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 83ms/step - loss: 22.5094 - mae: 22.9729 - val_loss: 27.0238 - val_mae: 27.4889 - learning_rate: 0.0010
Epoch 34/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 134ms/step - loss: 22.4113 - mae: 22.8746 - val_loss: 23.4580 - val_mae: 23.9212 - learning_rate: 0.0010
Epoch 35/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 85ms/step - loss: 22.0155 - mae: 22.4795 - val_loss: 21.1333 - val_mae: 21.5928 - learning_rate: 0.0010
Epoch 36/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - loss: 21.8398 - mae: 22.3016 - val_loss: 21.1406 - val_mae: 21.6049 - learning_rate: 0.0010
Epoch 37/60
30/31 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - loss: 21.2365 - mae: 21.6985

31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 105ms/step - loss: 21.8770 - mae: 22.3401 - val_loss: 17.7921 - val_mae: 18.2509 - learning_rate: 0.0010
Epoch 38/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - loss: 21.0702 - mae: 21.5312 - val_loss: 19.7996 - val_mae: 20.2606 - learning_rate: 0.0010
Epoch 39/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 4s 136ms/step - loss: 22.9666 - mae: 23.4255 - val_loss: 18.9524 - val_mae: 19.4084 - learning_rate: 0.0010
Epoch 40/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 87ms/step - loss: 20.5001 - mae: 20.9602 - val_loss: 22.4157 - val_mae: 22.8733 - learning_rate: 0.0010
Epoch 41/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 98ms/step - loss: 20.5724 - mae: 21.0324 - val_loss: 19.1966 - val_mae: 19.6522 - learning_rate: 0.0010
Epoch 42/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - loss: 19.8907 - mae: 20.3515 - val_loss: 20.6198 - val_mae: 21.0807 - learning_rate: 0.0010
Epoch 43/60
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 113ms/step - loss: 20.5217 - mae: 20.9818 - val_loss: 21.6348 - val_mae: 22.0919 - learning

In [ ]:
# График обучения
plot_training_history(history)

# Метрики и scatter-plot на тесте (P02)
metrics = evaluate_cnn(model, X_test, y_test, meta_test)


Метрики на тесте:
  MAE  = 51.65 мс
  RMSE = 65.72 мс
  R²   = -0.319


In [ ]:
# Достаточно этого перед обучением новой модели
from tensorflow.keras import backend as K
K.clear_session()

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

# Перемешиваем все биты случайно (пациенты в обеих частях)
idx = np.arange(len(y_clean))
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, random_state=42,
    stratify=[m['patient_id'] for m in meta_clean]  # равное представление
)

X_train, X_test = X_clean[train_idx], X_clean[test_idx]
y_train, y_test = y_clean[train_idx], y_clean[test_idx]
meta_test_new   = [meta_clean[i] for i in test_idx]

print(f'Train: {len(y_train)} битов')
print(f'Test:  {len(y_test)} битов')

# Обучаем заново
from tensorflow.keras import backend as K
K.clear_session()

from pwv_cnn import build_cnn
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

model2 = build_cnn(input_shape=(X_clean.shape[1], 3))
model2.compile(optimizer=Adam(learning_rate=1e-3), loss='huber', metrics=['mae'])

callbacks = [
    EarlyStopping(patience=12, restore_best_weights=True, monitor='val_mae'),
    ReduceLROnPlateau(factor=0.5, patience=6, min_lr=1e-5, monitor='val_mae'),
    ModelCheckpoint(f'{DATA_DIR}/pwv_cnn_v2.h5', save_best_only=True, monitor='val_mae'),
]

history2 = model2.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=60, batch_size=32,
    callbacks=callbacks, verbose=1,
)

Train: 1028 битов
Test:  258 битов
Epoch 1/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - loss: 125.5957 - mae: 126.0822

28/28 ━━━━━━━━━━━━━━━━━━━━ 16s 240ms/step - loss: 114.3353 - mae: 114.8215 - val_loss: 119.9328 - val_mae: 120.4184 - learning_rate: 0.0010
Epoch 2/60
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 278ms/step - loss: 59.1142 - mae: 59.5986

28/28 ━━━━━━━━━━━━━━━━━━━━ 8s 286ms/step - loss: 50.9741 - mae: 51.4576 - val_loss: 42.9583 - val_mae: 43.4387 - learning_rate: 0.0010
Epoch 3/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 140ms/step - loss: 43.8473 - mae: 44.3287 - val_loss: 45.1703 - val_mae: 45.6486 - learning_rate: 0.0010
Epoch 4/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 176ms/step - loss: 41.8982 - mae: 42.3795

28/28 ━━━━━━━━━━━━━━━━━━━━ 6s 232ms/step - loss: 41.7469 - mae: 42.2275 - val_loss: 40.7668 - val_mae: 41.2404 - learning_rate: 0.0010
Epoch 5/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 134ms/step - loss: 38.3142 - mae: 38.7957 - val_loss: 42.4102 - val_mae: 42.8896 - learning_rate: 0.0010
Epoch 6/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - loss: 39.7418 - mae: 40.2202 - val_loss: 46.7852 - val_mae: 47.2646 - learning_rate: 0.0010
Epoch 7/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - loss: 34.4299 - mae: 34.9100 - val_loss: 41.0547 - val_mae: 41.5351 - learning_rate: 0.0010
Epoch 8/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - loss: 33.2096 - mae: 33.6870 - val_loss: 42.5130 - val_mae: 42.9902 - learning_rate: 0.0010
Epoch 9/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - loss: 32.0430 - mae: 32.5212

28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 166ms/step - loss: 31.4684 - mae: 31.9447 - val_loss: 33.0777 - val_mae: 33.5542 - learning_rate: 0.0010
Epoch 10/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 82ms/step - loss: 32.1524 - mae: 32.6302 - val_loss: 43.3481 - val_mae: 43.8254 - learning_rate: 0.0010
Epoch 11/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - loss: 30.8088 - mae: 31.2862 - val_loss: 35.9416 - val_mae: 36.4179 - learning_rate: 0.0010
Epoch 12/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step - loss: 29.2797 - mae: 29.7558 - val_loss: 34.2992 - val_mae: 34.7705 - learning_rate: 0.0010
Epoch 13/60
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 29.0397 - mae: 29.5155

...

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 103ms/step - loss: 25.7828 - mae: 26.2531 - val_loss: 30.2384 - val_mae: 30.7091 - learning_rate: 0.0010
Epoch 19/60
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step - loss: 27.1608 - mae: 27.6317

28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 144ms/step - loss: 26.7330 - mae: 27.2042 - val_loss: 30.1359 - val_mae: 30.6077 - learning_rate: 0.0010
Epoch 20/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 105ms/step - loss: 25.1250 - mae: 25.5967 - val_loss: 31.1229 - val_mae: 31.5936 - learning_rate: 0.0010
Epoch 21/60
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - loss: 26.1417 - mae: 26.6162

28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 110ms/step - loss: 25.7626 - mae: 26.2358 - val_loss: 29.7345 - val_mae: 30.2078 - learning_rate: 0.0010
Epoch 22/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 83ms/step - loss: 25.4445 - mae: 25.9165 - val_loss: 31.0996 - val_mae: 31.5743 - learning_rate: 0.0010
Epoch 23/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 118ms/step - loss: 25.6932 - mae: 26.1642 - val_loss: 29.9426 - val_mae: 30.4118 - learning_rate: 0.0010
Epoch 24/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 4s 129ms/step - loss: 23.9607 - mae: 24.4302 - val_loss: 35.3291 - val_mae: 35.7990 - learning_rate: 0.0010
Epoch 25/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - loss: 24.5468 - mae: 25.0168 - val_loss: 30.5293 - val_mae: 31.0023 - learning_rate: 0.0010
Epoch 26/60
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 25.1462 - mae: 25.6149

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - loss: 25.2662 - mae: 25.7359 - val_loss: 28.7359 - val_mae: 29.2086 - learning_rate: 0.0010
Epoch 27/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - loss: 24.4609 - mae: 24.9296 - val_loss: 32.4497 - val_mae: 32.9151 - learning_rate: 0.0010
Epoch 28/60
27/28 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - loss: 24.2381 - mae: 24.7052

28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 114ms/step - loss: 23.4387 - mae: 23.9063 - val_loss: 25.7849 - val_mae: 26.2545 - learning_rate: 0.0010
Epoch 29/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 104ms/step - loss: 24.0423 - mae: 24.5111 - val_loss: 30.8974 - val_mae: 31.3679 - learning_rate: 0.0010
Epoch 30/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 82ms/step - loss: 23.0267 - mae: 23.4931 - val_loss: 30.2211 - val_mae: 30.6929 - learning_rate: 0.0010
Epoch 31/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 100ms/step - loss: 23.0661 - mae: 23.5323 - val_loss: 33.1359 - val_mae: 33.6034 - learning_rate: 0.0010
Epoch 32/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 3s 91ms/step - loss: 22.8538 - mae: 23.3202 - val_loss: 29.9101 - val_mae: 30.3799 - learning_rate: 0.0010
Epoch 33/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 5s 87ms/step - loss: 22.6825 - mae: 23.1498 - val_loss: 30.1216 - val_mae: 30.5923 - learning_rate: 0.0010
Epoch 34/60
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 85ms/step - loss: 21.5648 - mae: 22.0274 - val_loss: 36.0406 - val_mae: 36.5121 - learning_

In [ ]:
from pwv_cnn import evaluate_cnn, plot_training_history
plot_training_history(history2)
metrics2 = evaluate_cnn(model2, X_test, y_test, meta_test_new)


Метрики на тесте:
  MAE  = 26.69 мс
  RMSE = 36.71 мс
  R²   = 0.510


In [ ]:
train_pids = set(meta_clean[i]['patient_id'] for i in train_idx)
y_train_vals = y_clean[train_idx]
baseline_pred = np.full(len(y_test), np.median(y_train_vals))
mae_baseline  = np.mean(np.abs(baseline_pred - y_test))

print(f'Baseline MAE: {mae_baseline:.1f} мс')
print(f'CNN MAE:      {metrics2["mae"]:.1f} мс')
print(f'Улучшение:    {mae_baseline - metrics2["mae"]:.1f} мс '
      f'({(1 - metrics2["mae"]/mae_baseline)*100:.0f}%)')

Baseline MAE: 44.8 мс
CNN MAE:      26.7 мс
Улучшение:    18.1 мс (40%)




---



---



---



---



In [ ]:
from pwn_cnn import run_inference
result = run_inference(
    model = meodel2,
    filepath = f'{DATA_DIR}'/data4ch_0_7.csv',
    distance_m = 0.57, # Расстояние между датиками пациента, м
    tkeo_factor = 0.8, # Порог для алгоритма поиска R-зубцов, оператор Teager-Kaiser
    patient_id = 'P01', # Обозначение пациента
    show_plot - True,
)

SyntaxError: unterminated string literal (detected at line 4) (3561406014.py, line 4)

In [ ]:
# Все файлы сразу — сводная таблица
from pwv_cnn import run_inference_batch

df_inference = run_inference_batch(
    model       = model2,
    config_path = CONFIG,
    data_dir    = DATA_DIR,
    show_plot   = False,
)
display(df_inference)

ПОЛНЫЙ ПАЙПЛАЙН ПРИ УЖЕ СОХРАНЕННОЙ МОДЕЛИ

---

---





---



---



---



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/WORK/DATA'
CONFIG   = f'{DATA_DIR}/config.json'
sys.path.insert(0, DATA_DIR)

# Зависимости (если не установлены)
!pip install scipy plotly scikit-learn -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Ячейка 1 — загрузка модели с диска
import tensorflow as tf
import numpy as np

model2 = tf.keras.models.load_model(f'{DATA_DIR}/pwv_cnn_v2.h5')
print('Модель загружена:', model2.name)
print('Входная форма:', model2.input_shape)

Модель загружена: PWV_CNN
Входная форма: (None, 292, 3)


In [ ]:
# Ячейка 2 — батч-обработка (если нужны свежие результаты)
from batch_pwv import run_batch, plot_patient_comparison

df_signals, df_patients, df_features, df_good = run_batch(
    config_path=CONFIG, data_dir=DATA_DIR, plot=False)

patient_meta = {
    'P01': {'age': 26, 'notes': 'аритмия, лишний вес'},
    'P02': {'age': 40, 'notes': 'норма'},
    'P03': {'age': 55, 'notes': 'норма, жен.'},
}
plot_patient_comparison(df_features, patient_meta)


############################################################
  ПАЦИЕНТ P01   dist=0.57 м   tkeo=0.8
  Файлов: 14
############################################################

Файл: /content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_0_1.csv
Строк: 24997, столбцов: 5
Длина записи: 51.2 с
Артефактов (маска): 26.0%
R-пиков найдено: 79
Foot груди: 49,  пар с рукой: 49

Результаты:
  Валидных пар: 49
  PTT медиана: 65.5 мс  std: 69.1 мс
  PWV медиана: 8.70 м/с  std: 8.98 м/с
  HR: 97.5 уд/мин  RMSSD: 154.5 мс  SDNN: 162.9 мс

Файл: /content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_0_2.csv
Строк: 31749, столбцов: 5
Длина записи: 65.0 с
Артефактов (маска): 30.9%
R-пиков найдено: 99
Foot груди: 65,  пар с рукой: 65

Результаты:
  Валидных пар: 65
  PTT медиана: 108.5 мс  std: 78.9 мс
  PWV медиана: 5.25 м/с  std: 10.89 м/с
  HR: 97.5 уд/мин  RMSSD: 152.6 мс  SDNN: 150.3 мс

Файл: /content/drive/MyDrive/Colab Notebooks/WORK/DATA/data4ch_0_3.csv
Строк: 32140, столбцов: 5
Длина запис

In [ ]:
import importlib, sys
for mod in ['pwv_cnn', 'pwv_single', 'batch_pwv']:
    if mod in sys.modules:
        del sys.modules[mod]
# СБРОС КЭША ДЛЯ ЗАМЕНЫ МОДУЛЕЙ
from pwv_cnn import run_inference

In [ ]:
# Ячейка 3 — инференс на конкретном файле
from pwv_cnn import run_inference

result = run_inference(
    model       = model2,
    filepath    = f'{DATA_DIR}/data4ch_0_7.csv',
    distance_m  = 0.57,
    tkeo_factor = 0.8,
    patient_id  = 'P01',
    show_plot   = True,
)
 # Зелёные точки — алгоритмический метод (foot-to-foot)
 # Синие ромбы — CNN-предсказание


Инференс: data4ch_0_7.csv  [P01]
  Алгоритм:  PTT=98.3 мс  PWV=5.80 м/с  (n=64)
  CNN:       PTT=111.5 мс  PWV=5.11 м/с  (n=75)
  Расхождение алгоритм↔CNN: 13.2 мс


In [ ]:
import sys
for mod in ['pwv_cnn', 'pwv_single']:
    if mod in sys.modules: del sys.modules[mod]

from pwv_cnn import run_inference
import pandas as pd

demo_signals = [
    dict(model=model2,
         filepath=f'{DATA_DIR}/data4ch_0_7.csv',
         distance_m=0.57, tkeo_factor=0.8,
         patient_id='P01 (26л, аритмия)'),
    dict(model=model2,
         filepath=f'{DATA_DIR}/data4ch_1_3.csv',
         distance_m=0.52, tkeo_factor=0.6,
         patient_id='P02 (40л, норма)'),
    dict(model=model2,
         filepath=f'{DATA_DIR}/data4ch_4_3.csv',
         distance_m=0.38, tkeo_factor=0.4,
         patient_id='P03 (55л, норма)'),
]

results = []
for s in demo_signals:
    try:
        r = run_inference(show_plot=True, **s)
        if r: results.append(r)
    except Exception as e:
        print(f'  [!] Пропущен {s["patient_id"]}: {e}')

if results:
    df_demo = pd.DataFrame(results)[
        ['patient_id','algo_ptt_ms','cnn_ptt_ms',
         'agreement_ms','algo_pwv_ms','cnn_pwv_ms']]
    display(df_demo)


Инференс: data4ch_0_7.csv  [P01 (26л, аритмия)]
  Алгоритм:  PTT=98.3 мс  PWV=5.80 м/с  (n=64)
  CNN:       PTT=111.5 мс  PWV=5.11 м/с  (n=75)
  Расхождение алгоритм↔CNN: 13.2 мс



Инференс: data4ch_1_3.csv  [P02 (40л, норма)]
  Алгоритм:  PTT=73.7 мс  PWV=7.05 м/с  (n=143)
  CNN:       PTT=144.4 мс  PWV=3.60 м/с  (n=139)
  Расхождение алгоритм↔CNN: 70.7 мс



Инференс: data4ch_4_3.csv  [P03 (55л, норма)]
  Алгоритм:  PTT=106.5 мс  PWV=3.57 м/с  (n=92)
  CNN:       PTT=104.0 мс  PWV=3.65 м/с  (n=96)
  Расхождение алгоритм↔CNN: 2.5 мс


,patient_id,algo_ptt_ms,cnn_ptt_ms,agreement_ms,algo_pwv_ms,cnn_pwv_ms
0,"P01 (26л, аритмия)",98.3,111.5,13.2,5.80,5.11
1,"P02 (40л, норма)",73.7,144.4,70.7,7.05,3.60
2,"P03 (55л, норма)",106.5,104.0,2.5,3.57,3.65


Предварительные результаты по запуску сравнения пайплайн-алгоритма и CNN:

P01 (аритмия) — расхождение 14 мс:
Оба метода дают близкие результаты (~5–5.8 м/с). Разброс точек большой — это реальная beat-to-beat вариабельность от аритмии. CNN распределение чуть уже — модель слегка сглаживает выбросы.

P03 (норма, 55л) — расхождение 2 мс:
Лучший результат из трёх. Алгоритм и CNN практически совпадают, распределения перекрываются максимально. Это идеальная демонстрация согласованности методов.

P02 (норма, 40л) — расхождение 70 мс:
Здесь видна ключевая разница. Алгоритм даёт два кластера — ~20 мс (граничное значение, wrist-датчик не сработал) и ~200 мс (другая ошибка). Гистограмма алгоритма — два больших столбца по краям. CNN же выдаёт один стабильный кластер ~140–150 мс. Это именно тот случай, где CNN полезнее алгоритма.

Записи результатов: newplot (30), newplot (31), newplot (32)

In [ ]:
import importlib.util, sys

DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/WORK/DATA'

spec = importlib.util.spec_from_file_location(
    'pwv_bp_calibration',
    f'{DATA_DIR}/pwv_bp_calibration.py'
)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

sys.modules['pwv_bp_calibration'] = module

run_calibration = module.run_calibration
estimate_bp     = module.estimate_bp

# Это временное решение для ситуации, если модуль под оценку АД
# не загрузился, как в моём случае

In [ ]:
# Загрузка через importlib + калибровка — всё в одной ячейке
import importlib.util, sys
import numpy as np

DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/WORK/DATA'

spec = importlib.util.spec_from_file_location(
    'pwv_bp_calibration', f'{DATA_DIR}/pwv_bp_calibration.py')
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

run_calibration = mod.run_calibration
estimate_bp     = mod.estimate_bp

# Данные P01
pwv_values = [5.80, 5.35, 5.25, 8.70, 5.25, 7.32, 5.46]   # из df_signals P01
# СЮДА ВСТАВИТЬ РЕАЛЬНЫЕ ЗНАЧЕНИЯ АД С МАНЖЕТЫ, НЕ ЗАБЫТЬ О СООТВЕТСТВИИ
# ПОРЯДКА ВВОДА - СКОРОСТЬ ИЗ ФАЙЛА/СИСТОЛИЧЕСКОЕ/ДИАСТОЛИЧЕСКОЕ
sbp_values = [140, 136, 126, 142, 128, 142, 138]
dbp_values = [82, 90, 80, 93, 79, 92, 89]

calib = run_calibration(
    pwv_values = pwv_values,
    sbp_values = sbp_values,
    dbp_values = dbp_values,
    patient_id = 'P01',
    show_plot  = True,
)


Калибровка СРПВ → АД  [P01]
Точек измерений: 7
  СД (SBP): BP = 3.33×СРПВ + 115.48  R²=0.465  MAE=4.1 мм рт.ст.
  ДД (DBP): BP = 3.04×СРПВ + 67.67  R²=0.474  MAE=3.7 мм рт.ст.
  MAP: BP = 3.14×СРПВ + 83.60  R²=0.520  MAE=3.3 мм рт.ст.

  Корреляция Пирсона:
    СРПВ ↔ СД:  r = 0.682
    СРПВ ↔ ДД:  r = 0.689
    СРПВ ↔ MAP: r = 0.721


In [ ]:
df_bp = estimate_bp(calib, pwv_new=[4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 8.0])
display(df_bp)


Оценка АД по СРПВ [P01]:
 pwv_ms bp_str  map_est  in_range
    4.0 129/80     96.2      True
    4.5 130/81     97.7      True
    5.0 132/83     99.3      True
    5.5 134/84    100.9      True
    6.0 135/86    102.4      True
    6.5 137/87    104.0      True
    7.0 139/89    105.6      True
    8.0 142/92    108.7      True


,pwv_ms,sbp_est,dbp_est,map_est,bp_str,in_range
0,4.0,128.8,79.8,96.2,129/80,True
1,4.5,130.5,81.4,97.7,130/81,True
2,5.0,132.1,82.9,99.3,132/83,True
3,5.5,133.8,84.4,100.9,134/84,True
4,6.0,135.5,85.9,102.4,135/86,True
5,6.5,137.1,87.5,104.0,137/87,True
6,7.0,138.8,89.0,105.6,139/89,True
7,8.0,142.1,92.0,108.7,142/92,True


In [ ]:
# Оценка АД по медианным СРПВ всех 13 сигналов P01
pwv_signals = [8.70, 5.25, 3.90, 3.20, 4.22, 5.46,
               5.80, 7.32, 5.35, 3.89, 3.92, 4.97, 27.83]  # последний — артефакт

df_bp_signals = estimate_bp(calib, pwv_new=pwv_signals)
display(df_bp_signals)


Оценка АД по СРПВ [P01]:
 pwv_ms  bp_str  map_est  in_range
   8.70  144/94    110.9      True
   5.25  133/84    100.1      True
   3.90  128/80     95.9      True
   3.20  126/77     93.7      True
   4.22  130/81     96.9      True
   5.46  134/84    100.7      True
   5.80  135/85    101.8      True
   7.32  140/90    106.6      True
   5.35  133/84    100.4      True
   3.89  128/80     95.8      True
   3.92  129/80     95.9      True
   4.97  132/83     99.2      True
  27.83 208/152    171.0     False
[!] Часть значений вышла за физиологические границы — экстраполяция за пределы калибровки ненадёжна.


,pwv_ms,sbp_est,dbp_est,map_est,bp_str,in_range
0,8.70,144.5,94.2,110.9,144/94,True
1,5.25,133.0,83.7,100.1,133/84,True
2,3.90,128.5,79.5,95.9,128/80,True
3,3.20,126.1,77.4,93.7,126/77,True
4,4.22,129.5,80.5,96.9,130/81,True
5,5.46,133.7,84.3,100.7,134/84,True
6,5.80,134.8,85.3,101.8,135/85,True
7,7.32,139.9,90.0,106.6,140/90,True
8,5.35,133.3,84.0,100.4,133/84,True
9,3.89,128.4,79.5,95.8,128/80,True
